# 06c Cycle-index shift test
Phase 6c. Post-hoc and exploratory: **POST-HOC VARIANT, NOT A GATE RESULT**. Phase 6b found that adding log(cycle number) cuts Protocol D error. This notebook asks, for both the random forest and the MLP, whether that gain survives a shift to slow-aging cells, before the locked batch-3 test. Nothing here is tuned, nothing changes the primary model, batch 3 is never touched. The frozen gate, verdict, module and Phase 6b config are hash-checked at the start and the end. Writes only to `outputs/phase6c` (plus `src/soh_deep_v3.py`).

## 1. Guards

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, time, json, gc, math, hashlib, inspect, re, warnings, ctypes, difflib, importlib, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, psutil

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
T_START = time.time()

def keep_awake(on):
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 if on else 0x80000000)
    except Exception as e:
        print("keep-awake unavailable:", e)
keep_awake(True)

# ---- root and paths (ROOT = parent of notebooks/)
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "notebooks").is_dir() and (p / "data").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"cannot resolve ROOT (parent of notebooks/) from {_cwd}")
PROC = ROOT / "data/processed"; SPL = PROC / "splits"; P3 = ROOT / "outputs/phase3"; OUT4 = ROOT / "outputs/phase4"
OUT5B = ROOT / "outputs/phase5b"; OUT6B = ROOT / "outputs/phase6b"; SRC = ROOT / "src"
OUT = ROOT / "outputs/phase6c"; FIG = OUT / "figures"; CACHE = OUT / "cache"
for _p in (OUT, FIG, CACHE):
    _p.mkdir(parents=True, exist_ok=True)
P_GATE = OUT4 / "week2_gate.json"; P_MODULE = SRC / "soh_deep_v2.py"; P_MODULE3 = SRC / "soh_deep_v3.py"; P_VERDICT = OUT5B / "gate_verdict.json"
P_FZ = OUT6B / "frozen_config.json"; P_ABL = OUT6B / "ablation_results.csv"; P_EXTCELLS = OUT6B / "extreme_holdout_cells.csv"; P_EXTCOUNTS = OUT6B / "extreme_holdout_counts.csv"
P_CACHE6B = OUT6B / "cache"          # Phase 6b cache: read only

# ---- frozen facts
GATE_SHA = "40df2fb82096e994f8bf7b8084433bb5ca148d3fa92c780461f68c0c4247d901"
MODULE_SHA_EXPECTED = "fc53e37d506ebe51f6e4d71819c4b047de5bc38456669a160183531fbefebea6"
VERDICT_SHA = "2f59f82a7c7d8940636b55d7fd3397760d4486569d95575c802dbd02df69b1bb"
FZ_SHA_EXPECTED = "78b571c7526551c70a01cb709661052ddfb5b4c9783e4eb5babb1c7e11b65fe5"
EXPECTED_ROWS, EXPECTED_CELLS = 55963, 84
GATE_TARGET = {"W300": 0.7605, "W240": 0.5569}          # frozen gate targets; printed for information only
LABEL = "POST-HOC VARIANT, NOT A GATE RESULT"
NB_VERSION = "06c-v1"

# ---- checks (PASS / WARN / FAIL; FAIL raises)
CHECKS, WRITTEN, INTERVALS = [], [], {}
def dump_checks(partial=False, extra=None):
    n = lambda s: sum(c["level"] == s for c in CHECKS)
    rep = dict(phase="6c", label=LABEL, created=dt.datetime.now().isoformat(timespec="seconds"), root=str(ROOT), partial=partial,
               n_pass=n("PASS"), n_warn=n("WARN"), n_fail=n("FAIL"), intervals=INTERVALS, n_intervals=int(sum(INTERVALS.values())), checks=CHECKS)
    if extra: rep.update(extra)
    (OUT / "verification_report.json").write_text(json.dumps(rep, indent=2, default=str), encoding="utf-8")
def chk(level, name, detail=""):
    CHECKS.append(dict(level=level, name=name, detail=str(detail)))
    print(f"[{level}] {name}" + (f" | {detail}" if str(detail) else ""))
    if level == "FAIL":
        dump_checks(partial=True)
        raise AssertionError(f"{name}: {detail}")
def need(cond, name, detail="", level="FAIL"):
    chk("PASS" if cond else level, name, detail)
def ensure(cond, msg, detail=""):
    if not cond: chk("FAIL", msg, detail)
def require_cols(df, cols, name):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"{name}: missing column(s) {miss}; found {list(df.columns)}")
def add_int(section, n=1):
    INTERVALS[section] = INTERVALS.get(section, 0) + int(n)

# ---- guarded writes: nothing is ever written outside outputs/phase6c (the one exception, src/soh_deep_v3.py, has its own writer in section 2)
def _guard(p):
    p = Path(p).resolve()
    if OUT.resolve() not in p.parents:
        raise RuntimeError(f"refusing to write outside {OUT}: {p}")
    return p
def save_csv(df, name):
    p = _guard(OUT / name); df.to_csv(p, index=False); WRITTEN.append(p); return p
def save_text(text, name):
    p = _guard(OUT / name); p.write_text(text, encoding="utf-8"); WRITTEN.append(p); return p
def save_json(obj, name):
    p = _guard(OUT / name); p.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8"); WRITTEN.append(p); return p
def save_parquet(df, path):
    p = _guard(path); df.to_parquet(p); return p
def save_npy(path, arr):
    p = _guard(path); np.save(p, arr); return p

def ram_report(tag=""):
    av = psutil.virtual_memory().available / 1e9
    procs = []
    for p in psutil.process_iter(["name", "memory_info"]):
        try: procs.append((p.info["memory_info"].rss / 1e9, p.info["name"]))
        except Exception: pass
    procs.sort(reverse=True)
    print(f"RAM available {av:.1f} GB {tag}; top 8:", ", ".join(f"{n} {g:.1f}GB" for g, n in procs[:8]))
    if av < 2.0:
        chk("WARN", "RAM below 2 GB", f"{av:.1f} GB available; close memory hogs")
    return av
ram_report("(start)")

# ---- frozen-file hashes (start)
def sha_raw(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def sha_lf(p): return hashlib.sha256(Path(p).read_bytes().replace(b"\r\n", b"\n")).hexdigest()
for _p in (P_GATE, P_MODULE, P_VERDICT, P_FZ, P_ABL):
    need(_p.exists(), f"input exists: {_p.relative_to(ROOT)}")
def hash_state():
    return dict(gate=dict(sha=sha_raw(P_GATE), exp=GATE_SHA, mtime=P_GATE.stat().st_mtime_ns),
                module=dict(sha=sha_lf(P_MODULE), exp=MODULE_SHA_EXPECTED, mtime=P_MODULE.stat().st_mtime_ns),
                verdict=dict(sha=sha_raw(P_VERDICT), exp=VERDICT_SHA, mtime=P_VERDICT.stat().st_mtime_ns),
                frozen_config=dict(sha=sha_raw(P_FZ), exp=FZ_SHA_EXPECTED, mtime=P_FZ.stat().st_mtime_ns))
HASH0 = hash_state()
for _k, _v in HASH0.items():
    need(_v["sha"] == _v["exp"], f"sha256 matches frozen value at start: {_k}", _v["sha"][:16] + "...")
MODULE_SHA = HASH0["module"]["sha"]
_V = json.loads(P_VERDICT.read_text())
print("frozen verdict (read only, never recomputed): headline W300 PASS =", _V.get("headline_PASS"), "| companion W240 PASS =", _V.get("companion_PASS"))
need(_V.get("headline_PASS") is False and _V.get("companion_PASS") is True, "verdict file reads W300 FAIL / W240 PASS", level="WARN")
print(f"\n*** {LABEL} *** Nothing in this notebook is a gate result; nothing changes the primary model; the batch-3 test is not touched.\n")

# ---- module API (v2 is frozen; v3 is created in section 2)
sys.path.insert(0, str(SRC))
import soh_common as sc
import soh_deep_v2 as sd2
for _n in ("train_members", "fit_scalers", "predict", "sub_idx", "count_params", "make_single", "make_inputs"):
    need(hasattr(sd2, _n), f"soh_deep_v2.{_n} present")

# ---- CUDA, fp32, TF32 off, deterministic
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try: torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e: print("deterministic algorithms flag unavailable:", e)
# <<GPU>>
need(torch.cuda.is_available(), "CUDA available (needed for the training sections)")
_pr = torch.cuda.get_device_properties(0)
GPU_INFO = f"{_pr.name} {_pr.total_memory / 1e9:.1f} GB VRAM, torch {torch.__version__}"
DEVICE = torch.device("cuda")
# <</GPU>>
print("GPU:", GPU_INFO, "| TF32 matmul:", torch.backends.cuda.matmul.allow_tf32, "| cudnn deterministic:", torch.backends.cudnn.deterministic, "| dtype float32")
sc.assert_allowed(["q0"], roles=("primary",))
chk("PASS", "allow-list q0 (primary)")


RAM available 3.4 GB (start); top 8: Code.exe 0.7GB, Code.exe 0.7GB, MemCompression 0.7GB, Code.exe 0.6GB, chrome.exe 0.6GB, chrome.exe 0.5GB, claude.exe 0.4GB, MsMpEng.exe 0.4GB
[PASS] input exists: outputs\phase4\week2_gate.json
[PASS] input exists: src\soh_deep_v2.py
[PASS] input exists: outputs\phase5b\gate_verdict.json
[PASS] input exists: outputs\phase6b\frozen_config.json
[PASS] input exists: outputs\phase6b\ablation_results.csv
[PASS] sha256 matches frozen value at start: gate | 40df2fb82096e994...
[PASS] sha256 matches frozen value at start: module | fc53e37d506ebe51...
[PASS] sha256 matches frozen value at start: verdict | 2f59f82a7c7d8940...
[PASS] sha256 matches frozen value at start: frozen_config | 78b571c7526551c7...
frozen verdict (read only, never recomputed): headline W300 PASS = False | companion W240 PASS = True
[PASS] verdict file reads W300 FAIL / W240 PASS

*** POST-HOC VARIANT, NOT A GATE RESULT *** Nothing in this notebook is a gate result; nothing changes the 

In [2]:
SENS_CELLS = {"b1c8", "b1c10", "b1c12", "b1c13", "b1c22"}
STITCHED = [f"b1c{i}" for i in range(5)]
WINS = ["W300", "W240"]; GRID_V = {"W300": 3.00, "W240": 2.40}
B_BOOT, SEED_BOOT = 5000, 42
TERT_CUTS = (493, 710); TERT_LAB = ["short", "mid", "long"]

_lp = [p for p in (PROC / "labels_table_v2.parquet", SPL / "labels_table_v2.parquet", P3 / "labels_table_v2.parquet") if p.exists()]
need(bool(_lp), "labels_table_v2.parquet found")
labels = pd.read_parquet(_lp[0])
require_cols(labels, ["row_id", "cell_id", "batch", "cycle_number", "q0", "soh_nom", "scored2"], "labels_table_v2")
need(len(labels) == 104971, "labels rows", len(labels))
need(bool((labels["row_id"].values == np.arange(len(labels))).all()), "row_id == position")
cycles, qdlin, t_knots, registry = sc.load_tables(root=ROOT, mmap=True)
need(qdlin.shape == (len(labels), 1000), "qdlin shape", qdlin.shape)
qref = np.load(PROC / "qref_f32.npy"); qref_ids = pd.read_csv(PROC / "qref_cell_ids.csv")
require_cols(qref_ids, ["row", "cell_id"], "qref_cell_ids.csv")
need(qref.shape == (129, 1000) and bool((qref_ids["row"].values == np.arange(len(qref_ids))).all()), "qref shape/rows", qref.shape)
QREF_ROW = dict(zip(qref_ids["cell_id"], qref_ids["row"]))

pD = pd.read_csv(SPL / "protocol_D.csv")
require_cols(pD, ["repeat", "outer_fold", "cell_id", "role"], "protocol_D.csv")
need(set(pD["role"]) <= {"train", "test"}, "split roles")
DEV_CELLS = sorted(pD["cell_id"].unique())
need(len(DEV_CELLS) == EXPECTED_CELLS, "84 development cells", len(DEV_CELLS))
need(sum(c.startswith("b1") for c in DEV_CELLS) == 41 and sum(c.startswith("b2") for c in DEV_CELLS) == 43, "41 batch-1 + 43 batch-2")
need(not any(c.startswith("b3") for c in DEV_CELLS) and not (set(DEV_CELLS) & SENS_CELLS), "no batch-3 / sensitivity cells in development")
REPS = sorted(int(r) for r in pD["repeat"].unique()); FOLDS = sorted(int(f) for f in pD["outer_fold"].unique())
need((len(REPS), len(FOLDS)) == (3, 6), "3 repeats x 6 outer folds", (REPS, FOLDS))
_t = pD[pD["role"] == "test"]
need(bool((_t.groupby(["repeat", "cell_id"]).size() == 1).all()) and bool((_t.groupby("repeat")["cell_id"].nunique() == 84).all()), "each dev cell in exactly one test set per repeat")
_n = _t.groupby(["repeat", "outer_fold"]).size()
need(len(_n) == 18 and _n.min() >= 13 and _n.max() <= 17, "18 outer sets with 13-17 test cells", _n.tolist())

_life = pd.read_csv(P3 / "kept_cells_groups_strata.csv")
require_cols(_life, ["cell_id", "life"], "kept_cells_groups_strata.csv")
LIFE_MAP = dict(zip(_life["cell_id"], _life["life"]))
def _find_policy_file():
    for p in [P3 / "kept_cells_groups_strata.csv", *sorted(P3.glob("*.csv")), *sorted(SPL.glob("*.csv"))]:
        if not p.exists(): continue
        try: h = pd.read_csv(p, nrows=2)
        except Exception: continue
        if {"cell_id", "policy_norm"} <= set(h.columns): return p
    return None
_pp = _find_policy_file()
need(_pp is not None, "policy_norm found in a Phase 3 groups file", "looked in outputs/phase3/*.csv and data/processed/splits/*.csv for columns cell_id + policy_norm")
_pol = pd.read_csv(_pp).drop_duplicates("cell_id").set_index("cell_id")["policy_norm"]
print("policy_norm source:", _pp.relative_to(ROOT))
POLICY = {c: str(_pol.get(c)) for c in DEV_CELLS}
need(all(v not in ("None", "nan") for v in POLICY.values()), "policy_norm known for all 84 development cells")
print("distinct policy_norm among the 84 development cells:", len(set(POLICY.values())))

D = labels[labels["scored2"].astype(bool) & labels["cell_id"].isin(DEV_CELLS)]
D = D.sort_values(["cell_id", "cycle_number", "row_id"], kind="stable").reset_index(drop=True)
del labels; gc.collect()
need(len(D) == EXPECTED_ROWS, "development rows", len(D))
need(not D.duplicated(["cell_id", "cycle_number"]).any(), "(cell, cycle) unique")
D["y"] = 100.0 * D["soh_nom"].astype(float)
ROWID = D["row_id"].values.astype(np.int64); CELLID = D["cell_id"].values
Y = D["y"].values.astype(np.float64); Q0 = D["q0"].values.astype(np.float64); CYC = D["cycle_number"].values.astype(np.float64)
need(not (np.isnan(Y).any() or np.isnan(Q0).any() or np.isnan(CYC).any()), "no NaN in y / q0 / cycle_number")
need(bool((CYC >= 0).all()), "cycle numbers are non-negative (log and log1p defined)")
cells = D["cell_id"].unique(); NC, N, R = len(cells), len(D), len(REPS)
COUNTS = D.groupby("cell_id", sort=False).size().values; STARTS = np.r_[0, np.cumsum(COUNTS)[:-1]]
CELL_CODE = np.repeat(np.arange(NC), COUNTS)
need(list(cells) == sorted(cells) and COUNTS.sum() == N and bool((CELL_CODE == pd.factorize(D["cell_id"])[0]).all()), "cell blocks contiguous and sorted")
CELL_POS = {c: np.arange(s, s + k) for c, s, k in zip(cells, STARTS, COUNTS)}
CIDX = {c: i for i, c in enumerate(cells)}
BATCH = np.array([int(c[1]) for c in cells]); IS_ST = np.isin(cells, STITCHED)
LIFE = np.array([LIFE_MAP[c] for c in cells], dtype=float); LIFE_ROW = LIFE[CELL_CODE]
TERT = np.digitize(LIFE, TERT_CUTS)
print("life tertile sizes (cuts 493/710):", {TERT_LAB[k]: int((TERT == k).sum()) for k in range(3)}, "| life range", LIFE.min(), LIFE.max())
need(not (set(cells) & SENS_CELLS) and not any(str(c).startswith("b3") for c in cells), "cell arrays contain no batch-3 / sensitivity cells")
ROW_INDEX = pd.Index(ROWID)
LOGC = np.log1p(CYC)            # the extra MLP input: log1p of the row's OWN cycle number (before standardisation inside the module)

def eval_pos(cs):
    return np.concatenate([CELL_POS[c] for c in sorted(cs)])
def build_sets():
    S = []
    for r, f in sorted(pD[["repeat", "outer_fold"]].drop_duplicates().itertuples(index=False, name=None)):
        g = pD[(pD["repeat"] == r) & (pD["outer_fold"] == f)]
        tr = sorted(g[g["role"] == "train"]["cell_id"]); te = sorted(g[g["role"] == "test"]["cell_id"])
        need(not set(tr) & set(te) and len(tr) + len(te) == 84, f"train/test disjoint and complete ({r},{f})")
        need(not any(c.startswith("b3") or c in SENS_CELLS for c in tr + te), f"no b3/sensitivity ({r},{f})")
        S.append(dict(repeat=int(r), outer_fold=int(f), train=tr, test=te))
    return S
SETS = build_sets()
ram_report("(data)")


[PASS] labels_table_v2.parquet found
[PASS] labels rows | 104971
[PASS] row_id == position
[PASS] qdlin shape | (104971, 1000)
[PASS] qref shape/rows | (129, 1000)
[PASS] split roles
[PASS] 84 development cells | 84
[PASS] 41 batch-1 + 43 batch-2
[PASS] no batch-3 / sensitivity cells in development
[PASS] 3 repeats x 6 outer folds | ([0, 1, 2], [0, 1, 2, 3, 4, 5])
[PASS] each dev cell in exactly one test set per repeat
[PASS] 18 outer sets with 13-17 test cells | [14, 14, 14, 15, 14, 13, 14, 14, 13, 17, 13, 13, 14, 14, 14, 14, 14, 14]
[PASS] policy_norm found in a Phase 3 groups file | looked in outputs/phase3/*.csv and data/processed/splits/*.csv for columns cell_id + policy_norm
policy_norm source: outputs\phase3\kept_cells_groups_strata.csv
[PASS] policy_norm known for all 84 development cells
distinct policy_norm among the 84 development cells: 61
[PASS] development rows | 55963
[PASS] (cell, cycle) unique
[PASS] no NaN in y / q0 / cycle_number
[PASS] cycle numbers are non-negative

3.031601152

In [3]:
# ---- frozen config (Phase 6b, hash-checked above): read, never recomputed, never tuned
FZ = json.loads(P_FZ.read_text(encoding="utf-8")); COMMON, FROZEN_CFG = FZ["common"], FZ["windows"]
need(FZ["module_sha256"] == MODULE_SHA_EXPECTED and FZ["gate_sha256"] == GATE_SHA, "frozen_config.json records the frozen module and gate hashes")
SEEDS = [int(s) for s in COMMON["seeds"]]; E_EP = int(COMMON["E"])
need(SEEDS == [42, 43, 44, 45, 46] and E_EP == 120 and COMMON["batch"] == 256 and COMMON["stride"] == 5 and COMMON["head"] == "abs" and COMMON["huber_delta_pp"] == 1.0
     and COMMON["trees"] == 300 and COMMON["tree_max_features"] == 0.33 and COMMON["tree_seed"] == 42 and COMMON["noise_sigma"] == 0.02 and COMMON["clip"] == 1.0
     and COMMON["warmup_frac"] == 0.05 and COMMON["nominal_ah"] == 1.1, "frozen common config as specified (E 120, batch 256, seeds 42-46, stride 5, abs head, Huber 1 pp, 300 trees, max_features 0.33, tree seed 42)")
_exp_cfg = {"W300": (0.004, 1e-4, 3), "W240": (0.008, 1e-3, 1)}
for w in WINS:
    c = FROZEN_CFG[w]; need((c["lr"], c["wd"], c["b4r_leaf"]) == _exp_cfg[w], f"frozen window config {w}", f"lr {c['lr']}, wd {c['wd']}, B4r leaf {c['b4r_leaf']}")
FZ_SHA = HASH0["frozen_config"]["sha"]
print("frozen_config.json sha256:", FZ_SHA)

# ---- statistics machinery (as 06a / 06b)
def seg_mean(x):
    return np.add.reduceat(x, STARTS, axis=-1) / COUNTS
_WB = {}
def W_for(n, B=B_BOOT, seed=SEED_BOOT):
    if n not in _WB: _WB[n] = np.random.default_rng(seed).multinomial(n, np.full(n, 1.0 / n), size=B).astype(float)
    return _WB[n]
def wmean(W, M):
    ok = ~np.isnan(M); Z = np.where(ok, M, 0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        return (W @ Z) / (W @ ok.astype(float))
def ci95(draws):
    return np.nanpercentile(draws, [2.5, 97.5], axis=0)
W_ALL = W_for(NC)
def paired(d, section):
    """paired cluster bootstrap over cells for a vector of per-cell differences; returns (mean, lo, hi)"""
    lo, hi = ci95(wmean(W_for(len(d)), np.asarray(d, float))); add_int(section)
    return float(np.mean(d)), float(lo), float(hi)
def variant_metrics(P):
    ae = np.abs(P - Y); cell = seg_mean(ae).mean(0); tail = (CYC >= 0.8 * LIFE_ROW)
    num = np.add.reduceat(ae * tail, STARTS, axis=1); den = np.add.reduceat(tail.astype(float), STARTS)
    with np.errstate(invalid="ignore", divide="ignore"): last20 = np.nanmean(num / den)
    return dict(cell=cell, macro=float(cell.mean()), macro_excl_stitched=float(cell[~IS_ST].mean()), last20_macro=float(last20), bias=float((P - Y).mean()))

# ---- adapters (as 06b; the only addition is the optional cycle-index input for the +idx MLP)
from sklearn.ensemble import RandomForestRegressor
NJ = max(1, min(10, os.cpu_count() or 1)); CUR = {}; MEMBER_AUDIT = dict(members=0, trees=0)
_SRC2 = inspect.getsource(sd2); CFG_KEYS = sorted(set(re.findall(r'cfg\[\s*["\'](\w+)["\']\s*\]', _SRC2)))
print("cfg keys used by the module:", CFG_KEYS)
_DQKEY = hashlib.sha1(ROWID.tobytes()).hexdigest()[:10]
def get_dq(win):
    i_close = int(sc.grid_index(GRID_V[win])); width = i_close + 1; fd = CACHE / f"dq_{win}_{_DQKEY}.npy"; f6 = P_CACHE6B / f"dq_{win}_{_DQKEY}.npy"
    dq = None
    for src in (fd, f6):
        if src.exists():
            try:
                a = np.load(src)
                if a.shape == (N, width): dq = a; break
            except Exception: pass
    if dq is None:
        dq = np.zeros((N, width), np.float32)
        for c in cells:
            pos = CELL_POS[c]; blk = np.asarray(qdlin[ROWID[pos]], dtype=np.float32)
            out = np.asarray(sc.build_delta_q(blk, qref[QREF_ROW[c]], 0, i_close))
            ensure(out.shape == (len(pos), width), f"build_delta_q shape {win} {c}", out.shape); dq[pos] = out
        save_npy(fd, dq)
    for r in np.random.default_rng(SEED_BOOT).choice(N, 3, replace=False):
        one = np.asarray(sc.build_delta_q(np.asarray(qdlin[ROWID[r]], dtype=np.float32), qref[QREF_ROW[CELLID[r]]], 0, i_close))
        need(np.abs(one.ravel() - dq[r]).max() < 1e-6 and abs(float(dq[r, 0])) < 1e-6 and dq.shape[1] == width, f"delta-Q check {win} row {r}")
    return dq
def use_window(win):
    CUR.clear(); gc.collect(); torch.cuda.empty_cache()
    dq = get_dq(win); width = dq.shape[1]; g = np.asarray(sd2.sub_idx(width)).astype(int)
    need(len(g) == 32, "sub_idx gives 32 grid points", len(g))
    CUR.update(win=win, dq=dq, width=width, grid=g, Xt=torch.as_tensor(dq, device=DEVICE, dtype=torch.float32),
               q0t=torch.as_tensor(Q0, device=DEVICE, dtype=torch.float32), yt=torch.as_tensor(Y, device=DEVICE, dtype=torch.float32),
               logc=torch.as_tensor(LOGC, device=DEVICE, dtype=torch.float32))
    ram_report(f"({win})")

def make_cfg(hp):
    v = dict(WD=hp["wd"], BATCH=int(COMMON["batch"]), CLIP=float(COMMON["clip"]), HUBER_DELTA=float(COMMON["huber_delta_pp"]), NOISE_SIGMA=float(COMMON["noise_sigma"]),
             NOMINAL_AH=float(COMMON["nominal_ah"]), WARMUP_FRAC=float(COMMON["warmup_frac"]))
    miss = [k for k in CFG_KEYS if k not in v]
    if miss: raise RuntimeError(f"cfg key(s) {miss} used by the module but not set in make_cfg")
    return v
def train_rows(cs, seed):
    # stride 5: every 5th scored row per cell with a random per-cell offset (exactly as 05b / 06b), cell-balanced weights
    rng = np.random.default_rng(seed); pos, w = [], []
    for c in sorted(cs):
        q = CELL_POS[c][int(rng.integers(0, 5))::5]; pos.append(q); w.append(np.full(len(q), 1.0 / len(q)))
    pos, w = np.concatenate(pos).astype(np.int64), np.concatenate(w).astype(np.float32)
    ensure(len(pos) > 0 and set(CELLID[pos]) <= set(cs), "train rows valid and inside training cells")
    return pos, w
def split_preds(out, lens):
    arrs = [np.asarray(a).reshape(-1) for a in out]
    ensure(len(arrs) == len(lens) and all(len(a) == L for a, L in zip(arrs, lens)), "prediction lengths match eval rows")
    return [a.astype(np.float64) for a in arrs]
_PARAM_CHECKED = set()
SD3 = {"mod": None}          # filled in section 2
def deep_chunk(win, hp, chunk, var):
    for m in chunk: ensure(not set(m["train"]) & set(m["eval"]), "training members never include their own evaluation cells", f"{m['tag']}")
    tr = [train_rows(m["train"], m["seed"]) for m in chunk]; ev = [eval_pos(m["eval"]) for m in chunk]; cfg = make_cfg(hp)
    mem = [dict(seed=m["seed"], train_rows=r, train_w=w, val_rows=e, val_codes=pd.factorize(CELLID[e])[0].astype(np.int64)) for m, (r, w), e in zip(chunk, tr, ev)]
    use_idx = bool(var["idx"])
    if use_idx:
        ensure(SD3["mod"] is not None, "soh_deep_v3 loaded before the +idx MLP is trained")
        mod, kw = SD3["mod"], dict(extra=CUR["logc"])
    else:
        mod, kw = sd2, {}                                        # no-index MLP always runs on the frozen v2 module
    ret = mod.train_members(X=CUR["Xt"], q0=CUR["q0t"], y=CUR["yt"], members=mem, kind="M0", head="abs", lr=hp["lr"], epochs=E_EP, cfg=cfg, device=DEVICE,
                            dtype=torch.float32, log_every=10 ** 6, **kw)
    ensure(isinstance(ret, dict) and "val_pred" in ret, "train_members returned val_pred")
    preds = split_preds(ret["val_pred"], [len(e) for e in ev])
    if ret.get("nan_members"): chk("WARN", "NaN members", f"{win}/{var['name']}: members {list(ret['nan_members'])[:10]}")
    if use_idx not in _PARAM_CHECKED:
        _PARAM_CHECKED.add(use_idx); mk = (SD3["mod"] if use_idx else sd2).make_single
        npar = int(sd2.count_params(mk(kind="M0", L=CUR["width"], seed=0, **(dict(n_extra=1) if use_idx else {})))); exp = 3013 if use_idx else 2969
        need(npar == exp, f"count_params M0{'+idx' if use_idx else ''}", f"{npar} vs {exp}")
    MEMBER_AUDIT["members"] += len(chunk); del ret, mem; gc.collect(); torch.cuda.empty_cache()
    return preds
def run_chunked(fn, items, G, label):
    out, i = [], 0
    while i < len(items):
        ch = items[i:i + G]
        try: out += fn(ch); i += len(ch)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if G == 1: raise
            G = max(1, G // 2); chk("WARN", "CUDA OOM", f"{label}: retrying with G={G}")
    return out
def gcap(win): return 72 if win == "W300" else 36

def pred_frame(win, name, rep, of, seed, ep, pred):
    return pd.DataFrame(dict(window=win, variant=name, repeat=rep, outer_fold=of, seed=seed, row_id=ROWID[ep], cell_id=CELLID[ep], pred_pp=pred))
def sets_sig(sets):
    return [(s["repeat"], s["outer_fold"], hashlib.sha1("|".join(s["train"] + ["#"] + s["test"]).encode()).hexdigest()[:10]) for s in sets]
def ckey(kind, win, spec, hp, seeds, sets, module_sha):
    d = dict(module=module_sha, nb=NB_VERSION, kind=kind, win=win, spec=spec, hp=hp, seeds=list(seeds), E=E_EP, sets=sets_sig(sets), trees=COMMON["trees"])
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:20]
def run_deep(win, var, sets, tag):
    hp = FROZEN_CFG[win]; msha = SD3["sha"] if var["idx"] else MODULE_SHA
    fp = CACHE / f"{tag}_deep_{win}_{var['name']}_{ckey('deep', win, var, hp, SEEDS, sets, msha)}.parquet"
    if fp.exists():
        print(f"  cached: {fp.name}"); return pd.read_parquet(fp)
    t0 = time.time()
    mem = [dict(train=s["train"], eval=s["test"], seed=sd_, tag=(s["repeat"], s["outer_fold"])) for s in sets for sd_ in SEEDS]
    preds = run_chunked(lambda ch: deep_chunk(win, hp, ch, var), mem, gcap(win), f"{win}/{var['name']}")
    fr = pd.concat([pred_frame(win, var["name"], m["tag"][0], m["tag"][1], m["seed"], eval_pos(m["eval"]), p) for m, p in zip(mem, preds)], ignore_index=True)
    need(bool(np.isfinite(fr.pred_pp.values).all()), f"no NaN/inf predictions: {win}/{var['name']}")
    save_parquet(fr, fp); print(f"  trained {win}/{var['name']}: {len(mem)} members in {time.time() - t0:.0f}s"); return fr

TREE_X = {}
def tree_X(win, logcycle):
    k = (win, logcycle)
    if k not in TREE_X:
        X = np.column_stack([CUR["dq"][:, CUR["grid"]], Q0]); TREE_X[k] = np.column_stack([X, np.log(CYC)]) if logcycle else X     # log(cycle_number) exactly as 06b B4r:+log_cycle
    return TREE_X[k]
def tree_fit_predict(win, leaf, train_cs, eval_cs, seed, logcycle):
    ensure(not set(train_cs) & set(eval_cs), "tree training cells exclude their evaluation cells")
    pos, w = train_rows(train_cs, seed); X = tree_X(win, logcycle); off = 100.0 * Q0 / 1.1
    est = RandomForestRegressor(n_estimators=COMMON["trees"], min_samples_leaf=leaf, max_features=COMMON["tree_max_features"], n_jobs=NJ, random_state=seed)
    est.fit(X[pos], (Y - off)[pos], sample_weight=w); ep = eval_pos(eval_cs); MEMBER_AUDIT["trees"] += 1
    return ep, est.predict(X[ep]) + off[ep]
def run_tree(win, name, logcycle, sets, tag):
    leaf = FROZEN_CFG[win]["b4r_leaf"]; fp = CACHE / f"{tag}_tree_{win}_{name}_{ckey('tree', win, dict(name=name, leaf=leaf, log=logcycle), dict(leaf=leaf), [COMMON['tree_seed']], sets, MODULE_SHA)}.parquet"
    if fp.exists():
        print(f"  cached: {fp.name}"); return pd.read_parquet(fp)
    t0 = time.time(); out = []
    for s in sets:
        ep, p = tree_fit_predict(win, leaf, s["train"], s["test"], COMMON["tree_seed"], logcycle); out.append(pred_frame(win, name, s["repeat"], s["outer_fold"], COMMON["tree_seed"], ep, p))
    fr = pd.concat(out, ignore_index=True); need(bool(np.isfinite(fr.pred_pp.values).all()), f"no NaN/inf predictions: {win}/{name}")
    save_parquet(fr, fp); print(f"  fitted {win}/{name}: {len(sets)} forests in {time.time() - t0:.0f}s"); return fr

# ---- validation of saved prediction frames (Phase 6b cache is reused only if it covers exactly the expected members)
_FR_COLS = ["window", "variant", "repeat", "outer_fold", "seed", "row_id", "cell_id", "pred_pp"]
def frame_ok(fr, sets, seeds):
    if any(c not in fr.columns for c in _FR_COLS): return False, "missing columns"
    if not np.isfinite(fr.pred_pp.values).all(): return False, "non-finite predictions"
    pos = ROW_INDEX.get_indexer(fr.row_id.values)
    if (pos < 0).any() or not (cells[CELL_CODE[pos]] == fr.cell_id.values).all(): return False, "row_id / cell_id do not match the development rows"
    exp_n = 0
    for s in sets:
        g = fr[(fr["repeat"] == s["repeat"]) & (fr["outer_fold"] == s["outer_fold"])]
        if set(g.cell_id) != set(s["test"]): return False, f"cells of set ({s['repeat']},{s['outer_fold']}) differ from the expected test cells"
        if set(g.seed.astype(int)) != set(seeds): return False, f"seeds of set ({s['repeat']},{s['outer_fold']}) differ"
        if len(g) != len(seeds) * int(sum(COUNTS[CIDX[c]] for c in s["test"])) or g.groupby(["seed", "row_id"]).size().max() != 1: return False, f"set ({s['repeat']},{s['outer_fold']}) is not one prediction per row and seed"
        exp_n += len(g)
    if len(fr) != exp_n: return False, "extra rows outside the expected sets"
    return True, ""
def load_6b(pattern, sets, seeds):
    for p in sorted(P_CACHE6B.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True):
        try: fr = pd.read_parquet(p)
        except Exception: continue
        ok_, why = frame_ok(fr, sets, seeds)
        if ok_: return fr, p
        print(f"  6b cache file {p.name} not usable: {why}")
    return None, None
def ens_array(fr, nseeds, label):
    pos = ROW_INDEX.get_indexer(fr.row_id.values); ensure(bool((pos >= 0).all()), f"row_ids are development rows ({label})")
    arr = np.full((R, N), np.nan); rv = fr["repeat"].values
    for ri, r in enumerate(REPS):
        m = rv == r; p = pos[m]; s_ = np.bincount(p, weights=fr.pred_pp.values[m], minlength=N); n_ = np.bincount(p, minlength=N)
        ensure(bool(np.all(n_ == nseeds)), f"every row predicted exactly once per seed in repeat {r} ({label})", f"counts {np.unique(n_)}")
        arr[ri] = s_ / n_
    ensure(arr.shape == (R, N) and not np.isnan(arr).any(), f"ensemble array complete ({label})")
    for (r, f), g in fr.groupby(["repeat", "outer_fold"]):
        te = set(pD[(pD.repeat == r) & (pD.outer_fold == f) & (pD.role == "test")].cell_id); ensure(set(g.cell_id) == te, f"predictions cover exactly the test cells of ({r},{f}) ({label})")
    return arr
def ens_vec(fr, nseeds, label):
    pos = ROW_INDEX.get_indexer(fr.row_id.values); ensure(bool((pos >= 0).all()), f"row_ids are development rows ({label})"); vec = np.full(N, np.nan)
    s_ = np.bincount(pos, weights=fr.pred_pp.values, minlength=N); n_ = np.bincount(pos, minlength=N); m = n_ > 0
    ensure(bool(np.all(n_[m] == nseeds)), f"every held-out row predicted exactly once per seed ({label})"); vec[m] = s_[m] / n_[m]; return vec
def repro_gate(diff, from_cache, kind, name, detail):
    if diff <= 1e-6: chk("PASS", name, detail)
    elif from_cache or kind == "tree": chk("FAIL", name, detail)
    elif diff <= 0.05: chk("WARN", name, detail + " | MLP re-run on this machine drifted from the 6b numbers")
    else: chk("FAIL", name, detail)
chk("PASS", "adapters defined", f"seeds {SEEDS}, E={E_EP}, trees {COMMON['trees']}, tree jobs {NJ}")


[PASS] frozen_config.json records the frozen module and gate hashes
[PASS] frozen common config as specified (E 120, batch 256, seeds 42-46, stride 5, abs head, Huber 1 pp, 300 trees, max_features 0.33, tree seed 42)
[PASS] frozen window config W300 | lr 0.004, wd 0.0001, B4r leaf 3
[PASS] frozen window config W240 | lr 0.008, wd 0.001, B4r leaf 1
frozen_config.json sha256: 78b571c7526551c70a01cb709661052ddfb5b4c9783e4eb5babb1c7e11b65fe5
cfg keys used by the module: ['BATCH', 'CLIP', 'HUBER_DELTA', 'NOISE_SIGMA', 'NOMINAL_AH', 'WARMUP_FRAC', 'WD']
[PASS] adapters defined | seeds [42, 43, 44, 45, 46], E=120, trees 300, tree jobs 10


In [4]:
# ---- reproduction of the Phase 6b reference rows (M0:ref, B4r:ref, B4r:+log_cycle) from the 6b cache, within 1e-6; re-run with the frozen config if the cache is missing
ABL = pd.read_csv(P_ABL); require_cols(ABL, ["window", "variant", "macro_mae", "macro_excl_stitched", "last20_macro_mae", "pooled_bias"], "ablation_results.csv")
VAR_REF = dict(name="ref", idx=False); VAR_IDX = dict(name="idx", idx=True)
REF_SPECS = [("M0", "M0:ref", "deep", "abl_deep_{w}_ref_*.parquet", SEEDS), ("B4r", "B4r:ref", "tree", "abl_tree_{w}_ref_*.parquet", [COMMON["tree_seed"]]),
             ("B4r+idx", "B4r:+log_cycle", "tree", "abl_tree_{w}_+log_cycle_*.parquet", [COMMON["tree_seed"]])]
PD_ARR, PD_SRC, REPRO = {}, {}, []
for w in WINS:
    print(f"\n=== reference arrays {w} ===")
    for model, vname, kind, pat, seeds in REF_SPECS:
        fr, p = load_6b(pat.format(w=w), SETS, seeds)
        if fr is not None:
            src = f"6b cache: {p.name}"; from_cache = True; print(f"  {model}: {src}")
        else:
            src = "re-run with frozen config (6b cache missing or unusable)"; from_cache = False; print(f"  {model}: {src}")
            use_window(w)
            fr = run_deep(w, VAR_REF, SETS, "pd") if kind == "deep" else run_tree(w, "ref" if model == "B4r" else "log_cycle", model == "B4r+idx", SETS, "pd")
        arr = ens_array(fr, len(seeds), f"{w} {model}"); PD_ARR[(w, model)] = arr; PD_SRC[(w, model)] = src
        vm = variant_metrics(arr); ref = ABL[(ABL.window == w) & (ABL.variant == vname)]
        need(len(ref) == 1, f"ablation_results.csv has {w} {vname}", len(ref)); ref = ref.iloc[0]
        for met, mine, col in (("macro_mae", vm["macro"], "macro_mae"), ("macro_excl_stitched", vm["macro_excl_stitched"], "macro_excl_stitched"),
                               ("last20_macro_mae", vm["last20_macro"], "last20_macro_mae"), ("pooled_bias", vm["bias"], "pooled_bias")):
            diff = abs(mine - float(ref[col])); REPRO.append(dict(window=w, model=model, variant_6b=vname, metric=met, recomputed=mine, ablation_results=float(ref[col]), abs_diff=diff, source=src))
            repro_gate(diff, from_cache, kind, f"reproduces ablation_results.csv {w} {vname} {met}", f"{mine:.6f} vs {float(ref[col]):.6f} (|diff| {diff:.1e})")
    gc.collect(); torch.cuda.empty_cache()
REPRO = pd.DataFrame(REPRO); save_csv(REPRO, "reproduction_check.csv")



=== reference arrays W300 ===
  M0: 6b cache: abl_deep_W300_ref_7a70aa6b067792dd5e67.parquet
[PASS] ablation_results.csv has W300 M0:ref | 1
[PASS] reproduces ablation_results.csv W300 M0:ref macro_mae | 0.832337 vs 0.832337 (|diff| 0.0e+00)
[PASS] reproduces ablation_results.csv W300 M0:ref macro_excl_stitched | 0.697297 vs 0.697297 (|diff| 0.0e+00)
[PASS] reproduces ablation_results.csv W300 M0:ref last20_macro_mae | 1.595383 vs 1.595383 (|diff| 0.0e+00)
[PASS] reproduces ablation_results.csv W300 M0:ref pooled_bias | 0.572722 vs 0.572722 (|diff| 0.0e+00)
  B4r: 6b cache: abl_tree_W300_ref_314744ee591a60a1c141.parquet
[PASS] ablation_results.csv has W300 B4r:ref | 1
[PASS] reproduces ablation_results.csv W300 B4r:ref macro_mae | 0.816469 vs 0.816469 (|diff| 0.0e+00)
[PASS] reproduces ablation_results.csv W300 B4r:ref macro_excl_stitched | 0.739119 vs 0.739119 (|diff| 0.0e+00)
[PASS] reproduces ablation_results.csv W300 B4r:ref last20_macro_mae | 1.498868 vs 1.498868 (|diff| 0.0e+00)

WindowsPath('C:/workstation/3RD YEAR/Battery-Health-Estimation/outputs/phase6c/reproduction_check.csv')

## 2. MLP with cycle index
`src/soh_deep_v3.py` = v2 plus one optional extra scalar input per row (standardised log1p of the row's own cycle number, statistics from that member's training rows), concatenated with Q0: 34 inputs, 3,013 parameters. With the input off, v3 is numerically identical to v2.

In [5]:
# ---- soh_deep_v3.py = soh_deep_v2.py + ONE optional extra scalar input, produced by an exact-match patch of the frozen v2 (every replacement must match exactly once)
import hashlib
from pathlib import Path

V2_SHA_LF = "fc53e37d506ebe51f6e4d71819c4b047de5bc38456669a160183531fbefebea6"

V3_DOC = '''"""soh_deep_v3: soh_deep_v2 plus ONE optional extra scalar input per row (Phase 6c).

Generated by notebook 06c_cycle_index_shift_test.ipynb as an exact-match patch of the
frozen src/soh_deep_v2.py (which is never modified). The single change: an optional
extra scalar (the caller passes one value per data row, here log1p(cycle_number)),
standardised with mean / std of that member's TRAINING rows and concatenated with Q0
in front of the MLP (M0 then has 34 inputs, 3,013 parameters). With extra=None every
code path, parameter shape and random-number draw is identical to soh_deep_v2.
Do not edit by hand.
"""'''

V2_DOC_END = 'whenever its SHA-256 differs from the embedded source. Do not edit by hand.\n"""'

PATCHES = [
    # (label, old, new)
    ("docstring", None, None),     # handled separately: replaced from the first line to V2_DOC_END
    ("SingleM0.__init__",
     'class SingleM0(nn.Module):\n    def __init__(self, L):\n        super().__init__()\n        self.register_buffer("idx", torch.as_tensor(sub_idx(L)), persistent=False)\n        self.fc1 = nn.Linear(33, 44)\n',
     'def _qcols(q, nd):\n    """Per-row scalar columns: Q0 alone (as in v2) or Q0 plus the extra scalar."""\n    return q[..., None] if q.dim() == nd - 1 else q\n\n\nclass SingleM0(nn.Module):\n    def __init__(self, L, n_extra=0):\n        super().__init__()\n        self.register_buffer("idx", torch.as_tensor(sub_idx(L)), persistent=False)\n        self.fc1 = nn.Linear(33 + n_extra, 44)\n'),
    ("SingleM0.forward",
     '        x = torch.cat([d[:, self.idx], q[:, None]], 1)\n',
     '        x = torch.cat([d[:, self.idx], _qcols(q, 2)], 1)\n'),
    ("make_single",
     'def make_single(kind, L, seed, dtype=torch.float32):\n    torch.manual_seed(int(seed))\n    if kind == "M0":\n        m = SingleM0(L)\n',
     'def make_single(kind, L, seed, dtype=torch.float32, n_extra=0):\n    if n_extra and kind != "M0":\n        raise ValueError("the extra scalar input is implemented for M0 only")\n    torch.manual_seed(int(seed))\n    if kind == "M0":\n        m = SingleM0(L, n_extra)\n'),
    ("BatchedM0.forward",
     '        x = torch.cat([d[:, :, self.idx], q[:, :, None]], 2)\n',
     '        x = torch.cat([d[:, :, self.idx], _qcols(q, 3)], 2)\n'),
    ("fit_scalers signature",
     'def fit_scalers(X, q0, y, rows_list, head, nominal, device, dtype):\n',
     'def fit_scalers(X, q0, y, rows_list, head, nominal, device, dtype, extra=None):\n'),
    ("fit_scalers init",
     '    s, qm, qs, ym, ys = [], [], [], [], []\n',
     '    s, qm, qs, ym, ys = [], [], [], [], []\n    em, es = [], []\n'),
    ("fit_scalers extra stats",
     '        qs.append(float(qv.std()))\n',
     '        qs.append(float(qv.std()))\n        if extra is not None:\n            ev = extra[rt].double()\n            em.append(float(ev.mean()))\n            es.append(float(ev.std()))\n'),
    ("fit_scalers output",
     '    out["ym"] = torch.tensor(ym, dtype=dtype, device=device)\n    return out\n',
     '    out["ym"] = torch.tensor(ym, dtype=dtype, device=device)\n    if extra is not None:\n        out["es"] = torch.tensor([x if x > 1e-12 else 1.0 for x in es], dtype=dtype, device=device)\n        out["em"] = torch.tensor(em, dtype=dtype, device=device)\n    return out\n'),
    ("make_inputs",
     'def make_inputs(X, q0, idx, sc):\n    d = X[idx] / sc["s"][:, None, None]\n    q = (q0[idx] - sc["qm"][:, None]) / sc["qs"][:, None]\n    return d, q\n',
     'def make_inputs(X, q0, idx, sc, extra=None):\n    d = X[idx] / sc["s"][:, None, None]\n    q = (q0[idx] - sc["qm"][:, None]) / sc["qs"][:, None]\n    if extra is not None:\n        e = (extra[idx] - sc["em"][:, None]) / sc["es"][:, None]\n        q = torch.stack([q, e], 2)\n    return d, q\n'),
    ("predict signature",
     'def predict(model, X, q0, rows_list, sc, head, nominal, device, dtype, chunk=1024):\n',
     'def predict(model, X, q0, rows_list, sc, head, nominal, device, dtype, chunk=1024, extra=None):\n'),
    ("predict body",
     '        idx = pt[:, a:a + chunk]\n        d, q = make_inputs(X, q0, idx, sc)\n',
     '        idx = pt[:, a:a + chunk]\n        d, q = make_inputs(X, q0, idx, sc, extra)\n'),
    ("train_members signature",
     '                  dtype=torch.float32, log_every=10):\n',
     '                  dtype=torch.float32, log_every=10, extra=None):\n'),
    ("train_members singles",
     '    singles = [make_single(kind, L, m["seed"], dtype) for m in members]\n',
     '    n_extra = 0 if extra is None else 1\n    singles = [make_single(kind, L, m["seed"], dtype, n_extra) for m in members]\n'),
    ("train_members scalers",
     '    sc = fit_scalers(X, q0, y, train_rows, head, nominal, device, dtype)\n',
     '    sc = fit_scalers(X, q0, y, train_rows, head, nominal, device, dtype, extra)\n'),
    ("train_members loop inputs",
     '            d, q = make_inputs(X, q0, idx, sc)\n            if sigma > 0:\n',
     '            d, q = make_inputs(X, q0, idx, sc, extra)\n            if sigma > 0:\n'),
    ("train_members val predict (epoch)",
     '        if ep % log_every == 0 or ep == epochs:\n            vp = predict(model, X, q0, val_rows, sc, head, nominal, device, dtype)\n',
     '        if ep % log_every == 0 or ep == epochs:\n            vp = predict(model, X, q0, val_rows, sc, head, nominal, device, dtype, extra=extra)\n'),
    ("train_members val predict (final)",
     '\n    vp = predict(model, X, q0, val_rows, sc, head, nominal, device, dtype)\n    tp = predict(model, X, q0, train_rows, sc, head, nominal, device, dtype)\n',
     '\n    vp = predict(model, X, q0, val_rows, sc, head, nominal, device, dtype, extra=extra)\n    tp = predict(model, X, q0, train_rows, sc, head, nominal, device, dtype, extra=extra)\n'),
]


def v3_text_from_v2_bytes(raw):
    """raw: bytes of v2 (CRLF or LF). Returns (v2_text_lf, v3_text_lf, log)."""
    lf = raw.replace(b"\r\n", b"\n")
    sha = hashlib.sha256(lf).hexdigest()
    if sha != V2_SHA_LF:
        raise RuntimeError(f"soh_deep_v2.py hash {sha} != frozen {V2_SHA_LF}")
    t = lf.decode("utf-8"); v2 = t; log = []
    # docstring
    a = t.index('"""soh_deep_v2')
    b = t.index(V2_DOC_END, a) + len(V2_DOC_END)
    if t.count(V2_DOC_END) != 1 or a != 0:
        raise RuntimeError("docstring anchor not unique / not at start")
    t = V3_DOC + t[b:]; log.append(("docstring", 1))
    for label, old, new in PATCHES[1:]:
        n = t.count(old)
        if n != 1:
            raise RuntimeError(f"patch '{label}': expected exactly one match, found {n}")
        t = t.replace(old, new); log.append((label, n))
    return v2, t, log

# ---- build, write and load src/soh_deep_v3.py
V2_TEXT, V3_TEXT, PATCH_LOG = v3_text_from_v2_bytes(P_MODULE.read_bytes())          # raises unless v2 hashes to the frozen value and every patch matches exactly once
need(len(PATCH_LOG) == 18 and all(n == 1 for _, n in PATCH_LOG), "v3 patch: every replacement matched exactly once", f"{len(PATCH_LOG)} replacements")
V3_BYTES = V3_TEXT.encode("utf-8"); V3_SHA = hashlib.sha256(V3_BYTES).hexdigest()
_cur = P_MODULE3.read_bytes().replace(b"\r\n", b"\n") if P_MODULE3.exists() else None
if _cur != V3_BYTES:
    assert P_MODULE3.resolve() == (SRC / "soh_deep_v3.py").resolve() and P_MODULE3.resolve() != P_MODULE.resolve()
    P_MODULE3.write_bytes(V3_BYTES); print("wrote", P_MODULE3.relative_to(ROOT), "(new" if _cur is None else "(replaced a different version", ")")
else:
    print("src/soh_deep_v3.py already equals the patched text; left as is")
need(sha_lf(P_MODULE) == MODULE_SHA_EXPECTED, "soh_deep_v2.py untouched (sha256 unchanged after building v3)", sha_lf(P_MODULE)[:16] + "...")
need(sha_lf(P_MODULE3) == V3_SHA, "soh_deep_v3.py on disk equals the patched text")
print("soh_deep_v3.py sha256 (LF):", V3_SHA, "\nsoh_deep_v3.py sha256 (file bytes):", sha_raw(P_MODULE3))
_d = list(difflib.unified_diff(V2_TEXT.splitlines(), V3_TEXT.splitlines(), "soh_deep_v2.py", "soh_deep_v3.py", lineterm="", n=0))
print(f"diff v2 -> v3: {sum(l.startswith('+') and not l.startswith('+++') for l in _d)} lines added, {sum(l.startswith('-') and not l.startswith('---') for l in _d)} removed; every change belongs to the optional extra scalar input:\n" + "\n".join(_d))
if "soh_deep_v3" in sys.modules: del sys.modules["soh_deep_v3"]
importlib.invalidate_caches(); sd3 = importlib.import_module("soh_deep_v3")
need(Path(sd3.__file__).resolve() == P_MODULE3.resolve() and sd3.self_hash() == sha_raw(P_MODULE3), "soh_deep_v3 imported from src/")
SD3["mod"], SD3["sha"] = sd3, V3_SHA
need(sorted(set(re.findall(r'cfg\[\s*["\'](\w+)["\']\s*\]', inspect.getsource(sd3)))) == CFG_KEYS, "v3 uses exactly the same cfg keys as v2", CFG_KEYS)


[PASS] v3 patch: every replacement matched exactly once | 18 replacements
src/soh_deep_v3.py already equals the patched text; left as is
[PASS] soh_deep_v2.py untouched (sha256 unchanged after building v3) | fc53e37d506ebe51...
[PASS] soh_deep_v3.py on disk equals the patched text
soh_deep_v3.py sha256 (LF): 9c022dd391193de9c65f2d77b9dcdc5a2e939419596ecb38a55399ffcc06cbbb 
soh_deep_v3.py sha256 (file bytes): 9c022dd391193de9c65f2d77b9dcdc5a2e939419596ecb38a55399ffcc06cbbb
diff v2 -> v3: 45 lines added, 21 removed; every change belongs to the optional extra scalar input:
--- soh_deep_v2.py
+++ soh_deep_v3.py
@@ -1,4 +1,9 @@
-"""soh_deep_v2: deep models and batched trainer for Phase 5a2.
-
-OWNED by notebook 05a2_deep_pilot_fix.ipynb. The notebook overwrites this file
-whenever its SHA-256 differs from the embedded source. Do not edit by hand.
+"""soh_deep_v3: soh_deep_v2 plus ONE optional extra scalar input per row (Phase 6c).
+
+Generated by notebook 06c_cycle_index_shift_test.ipynb as

In [6]:
# ---- tests (a)-(d): CPU, float64, real development rows of W300
V3_CHECK_START = len(CHECKS)
import numpy as np
import torch


def v3_tests(sd2, sd3, X, q0, y, logc, cellcode, ok):
    """CPU float64 tests of soh_deep_v3 against the frozen soh_deep_v2. X (N,L), q0, y, logc (N,) arrays;
    cellcode (N,) int cell index per row. ok(cond, name, detail) records PASS/FAIL (FAIL raises)."""
    dev, f64 = torch.device("cpu"), torch.float64
    X, q0, y, e = (torch.as_tensor(np.asarray(a), dtype=f64, device=dev) for a in (X, q0, y, logc))
    L = X.shape[1]; cellcode = np.asarray(cellcode); cells_ = np.unique(cellcode)
    tr_cells, va_cells = cells_[:-3], cells_[-3:]
    rng = np.random.default_rng(0)
    tr = np.concatenate([np.where(cellcode == c)[0][int(rng.integers(0, 5))::5] for c in tr_cells]).astype(np.int64)
    va = np.concatenate([np.where(cellcode == c)[0] for c in va_cells]).astype(np.int64)
    cfg = dict(WD=1e-3, BATCH=64, CLIP=1.0, HUBER_DELTA=1.0, NOISE_SIGMA=0.02, NOMINAL_AH=1.1, WARMUP_FRAC=0.05)
    seeds = [42, 43, 44]
    mem = lambda: [dict(seed=s, train_rows=tr, train_w=np.ones(len(tr)) / len(tr), val_rows=va, val_codes=pd_factorize(cellcode[va])) for s in seeds]
    out = {}
    # (a1) same seed -> identical initial weights
    for s in seeds[:2]:
        a, b = sd2.make_single("M0", L, s, f64), sd3.make_single("M0", L, s, f64)
        ok(all(torch.equal(p, r) for p, r in zip(a.parameters(), b.parameters())), f"(a) v3 n_extra=0 initial weights identical to v2 (seed {s})")
    # (a2) full training, extra disabled: float64 curves and predictions equal to 1e-12
    r2 = sd2.train_members(X, q0, y, mem(), "M0", "abs", 4e-3, 3, cfg, dev, dtype=f64, log_every=1)
    r3 = sd3.train_members(X, q0, y, mem(), "M0", "abs", 4e-3, 3, cfg, dev, dtype=f64, log_every=1)
    d_curve = float(np.abs(r2["curve"]["val_mae"] - r3["curve"]["val_mae"]).max()); d_loss = float(np.abs(r2["curve"]["train_loss"] - r3["curve"]["train_loss"]).max())
    d_pred = max(float(np.abs(a.astype(np.float64) - b.astype(np.float64)).max()) for a, b in zip(r2["val_pred"], r3["val_pred"]))
    d_tm = float(np.abs(r2["train_mae"] - r3["train_mae"]).max())
    ok(max(d_curve, d_loss, d_tm) <= 1e-12 and d_pred <= 1e-12, "(a) extra disabled: v3 equals v2 to 1e-12 in float64 (validation curve, training loss, train MAE, predictions)",
       f"max |diff|: val curve {d_curve:.1e}, train loss {d_loss:.1e}, train MAE {d_tm:.1e}, predictions {d_pred:.1e}")
    out["a_max_diff"] = max(d_curve, d_loss, d_tm, d_pred)
    # (a3) scalers without extra identical, and no extra keys
    rl = [tr, tr[:50]]
    s2 = sd2.fit_scalers(X, q0, y, rl, "abs", 1.1, dev, f64); s3 = sd3.fit_scalers(X, q0, y, rl, "abs", 1.1, dev, f64)
    ok(set(s2) == set(s3) and all(torch.equal(s2[k], s3[k]) for k in s2), "(a) extra disabled: scalers identical and no extra statistics created")
    # (b) parameter counts
    n0 = sd3.count_params(sd3.make_single("M0", L, 42, f64)); n1 = sd3.count_params(sd3.make_single("M0", L, 42, f64, 1)); n2 = sd2.count_params(sd2.make_single("M0", L, 42, f64))
    ok(n0 == n2 == 2969 and n1 == 3013 and n1 - n0 == 44, "(b) parameter counts: M0 2,969; M0+idx 3,013 (= 2,969 + 44)", f"v2 {n2}, v3 no-idx {n0}, v3 +idx {n1}")
    sg = [sd3.make_single("M0", L, s, f64, 1) for s in seeds]; bm = sd3.make_batched("M0", L, sg)
    ok(sum(p.numel() for p in bm.parameters()) == 3 * 3013 and tuple(bm.W1.shape) == (3, 34, 44), "(b) batched M0+idx has 34 inputs per member", f"W1 {tuple(bm.W1.shape)}")
    out["n_params"] = (n0, n1)
    # (c) scaler statistics come from the member's training rows only
    sc_ = sd3.fit_scalers(X, q0, y, [tr, tr[:80]], "abs", 1.1, dev, f64, extra=e)
    for g, r in enumerate([tr, tr[:80]]):
        ev = e[torch.as_tensor(r)]
        ok(abs(float(sc_["em"][g]) - float(ev.mean())) <= 1e-12 and abs(float(sc_["es"][g]) - float(ev.std())) <= 1e-12, f"(c) member {g}: extra-input mean/std equal the training-row statistics")
    e2 = e.clone(); mask = np.ones(len(e), bool); mask[tr] = False; mask[tr[:80]] = False; e2[torch.as_tensor(np.where(mask)[0])] += 7.0
    sc2 = sd3.fit_scalers(X, q0, y, [tr, tr[:80]], "abs", 1.1, dev, f64, extra=e2)
    ok(torch.equal(sc_["em"], sc2["em"]) and torch.equal(sc_["es"], sc2["es"]), "(c) changing the extra values of rows outside the training rows leaves the statistics unchanged")
    sc_all = float(e.mean()); ok(abs(sc_all - float(sc_["em"][0])) > 1e-9, "(c) statistics differ from all-row statistics (the check is not vacuous)", f"{float(sc_['em'][0]):.5f} vs {sc_all:.5f}")
    # (d) each row's own extra value is what enters the model
    idx = torch.as_tensor(np.stack([va[:40], va[40:80]]))
    d0, q_ = sd3.make_inputs(X, q0, idx, sc_, e)
    exp = (e[idx] - sc_["em"][:, None]) / sc_["es"][:, None]
    ok(q_.shape == (2, 40, 2) and torch.equal(q_[..., 1], exp) and torch.equal(q_[..., 0], sd2.make_inputs(X, q0, idx, sc_)[1]), "(d) row input = Q0 scaled as in v2 plus its own standardised extra value")
    e3 = e.clone(); other = np.setdiff1d(np.arange(len(e)), idx.numpy().ravel()); e3[torch.as_tensor(other)] += 3.0
    _, q3 = sd3.make_inputs(X, q0, idx, sc_, e3); ok(torch.equal(q3, q_), "(d) other rows' extra values never reach a row's input")
    p0 = idx[0, 5].item(); e4 = e.clone(); e4[p0] += 3.0; _, q4 = sd3.make_inputs(X, q0, idx, sc_, e4)
    ch = (q4 != q_).any(2).nonzero().tolist(); ok(ch == [[0, 5]], "(d) changing one row's own extra value changes only that row's input", f"changed {ch}")
    bm2 = sd3.make_batched("M0", L, sg[:2]); rl2 = [va[:30], va[30:60]]; sc3 = sd3.fit_scalers(X, q0, y, [tr, tr], "abs", 1.1, dev, f64, extra=e)
    pa = sd3.predict(bm2, X, q0, rl2, sc3, "abs", 1.1, dev, f64, extra=e); pb = sd3.predict(bm2, X, q0, rl2, sc3, "abs", 1.1, dev, f64, extra=e3)
    pc = sd3.predict(bm2, X, q0, rl2, sc3, "abs", 1.1, dev, f64, extra=e4)
    ok(all(np.array_equal(a, b) for a, b in zip(pa, pb)), "(d) predictions of a row do not depend on other rows' extra values")
    ok(any(not np.array_equal(a, b) for a, b in zip(pa, pc)) or p0 not in np.concatenate(rl2), "(d) predictions react to the row's own extra value (not vacuous)")
    # forward parity: batched member vs its single model
    dd, qq = sd3.make_inputs(X, q0, idx, sc_, e); zb = bm(dd[:1].expand(3, -1, -1), qq[:1].expand(3, -1, -1))
    zs = torch.stack([m(dd[0], qq[0]) for m in sg]); ok(float((zb - zs).detach().abs().max()) <= 1e-12, "batched forward equals single-model forward (34 inputs)", f"{float((zb - zs).detach().abs().max()):.1e}")
    # +idx trains and is not identical to no-idx
    r4 = sd3.train_members(X, q0, y, mem(), "M0", "abs", 4e-3, 3, cfg, dev, dtype=f64, log_every=1, extra=e)
    ok(all(np.isfinite(v).all() for v in r4["val_pred"]) and not r4["nan_members"], "M0+idx trains without NaN")
    ok(max(float(np.abs(a - b).max()) for a, b in zip(r3["val_pred"], r4["val_pred"])) > 1e-6, "M0+idx predictions differ from M0 (extra input is used)")
    return out


def pd_factorize(a):
    import pandas as pd
    return pd.factorize(a)[0].astype(np.int64)

_dq64 = get_dq("W300"); _sel = [cells[i] for i in np.argsort(LIFE, kind="stable")[::7][:12]]; _rows = np.concatenate([CELL_POS[c] for c in _sel])
print(f"v3 tests run on {len(_sel)} real development cells ({len(_rows):,} rows, {_dq64.shape[1]} grid points), CPU, float64")
_ok = lambda c, n, d="": need(bool(c), n, d)
V3_TEST_OUT = v3_tests(sd2, sd3, _dq64[_rows], Q0[_rows], Y[_rows], LOGC[_rows], pd.factorize(CELLID[_rows])[0], _ok)
# (c) pipeline level: scalers of an actual outer training set come from its training cells only
_tr_rows = train_rows(SETS[0]["train"], 42)[0]; _Xc, _q0c, _yc, _lc = (torch.as_tensor(a, dtype=torch.float32) for a in (_dq64, Q0, Y, LOGC))
_s = sd3.fit_scalers(_Xc, _q0c, _yc, [_tr_rows], "abs", 1.1, torch.device("cpu"), torch.float32, extra=_lc)
_te_rows = eval_pos(SETS[0]["test"]); _man = LOGC[_tr_rows]
need(abs(float(_s["em"][0]) - _man.mean()) < 1e-5 and abs(float(_s["es"][0]) - _man.std(ddof=1)) < 1e-5, "(c) outer set (0,0): extra-input mean/std equal those of the training cells' rows",
     f"mean {float(_s['em'][0]):.4f}, std {float(_s['es'][0]):.4f}")
need(abs(float(_s["em"][0]) - LOGC[np.r_[_tr_rows, _te_rows]].mean()) > 1e-4 or abs(float(_s["es"][0]) - LOGC[np.r_[_tr_rows, _te_rows]].std(ddof=1)) > 1e-4, "(c) outer set (0,0): statistics differ from those that would include the test cells (not vacuous)")
# (d) on the real data: the index is the row's own cycle number
need(np.array_equal(LOGC, np.log1p(D["cycle_number"].values.astype(float))), "(d) extra input = log1p of each row's own cycle_number")
need(int(pd.DataFrame(dict(c=CYC, v=LOGC)).groupby("c")["v"].nunique().max()) == 1, "(d) the value depends on the cycle number only (same cycle in different cells / lives gives the same input)")
need(not np.allclose(LOGC, np.log1p(LIFE_ROW)) and not np.allclose(LOGC, np.log1p(CYC / LIFE_ROW)) and not np.allclose(LOGC, np.log1p(np.clip(LIFE_ROW - CYC, 0, None))), "(d) not a function of life, fraction of life or remaining life")
need(all(bool(np.all(np.diff(LOGC[CELL_POS[c]]) > 0)) for c in cells), "(d) strictly increasing with cycle inside every cell (available at the time of the measurement)")
V3_CHECKS = pd.DataFrame(CHECKS[V3_CHECK_START:])[["level", "name", "detail"]]; save_csv(V3_CHECKS, "v3_checks.csv")
print(f"\nsoh_deep_v3 verified: {len(V3_CHECKS)} checks, all PASS. sha256 {V3_SHA}")
del _dq64, _Xc, _q0c, _yc, _lc; gc.collect()


[PASS] delta-Q check W300 row 36631
[PASS] delta-Q check W300 row 4994
[PASS] delta-Q check W300 row 43312
v3 tests run on 12 real development cells (6,956 rows, 334 grid points), CPU, float64
[PASS] (a) v3 n_extra=0 initial weights identical to v2 (seed 42)
[PASS] (a) v3 n_extra=0 initial weights identical to v2 (seed 43)
[PASS] (a) extra disabled: v3 equals v2 to 1e-12 in float64 (validation curve, training loss, train MAE, predictions) | max |diff|: val curve 0.0e+00, train loss 0.0e+00, train MAE 0.0e+00, predictions 0.0e+00
[PASS] (a) extra disabled: scalers identical and no extra statistics created
[PASS] (b) parameter counts: M0 2,969; M0+idx 3,013 (= 2,969 + 44) | v2 2969, v3 no-idx 2969, v3 +idx 3013
[PASS] (b) batched M0+idx has 34 inputs per member | W1 (3, 34, 44)
[PASS] (c) member 0: extra-input mean/std equal the training-row statistics
[PASS] (c) member 1: extra-input mean/std equal the training-row statistics
[PASS] (c) changing the extra values of rows outside the trai

436

## 3. Protocol D, fixed config
M0+idx and B4r+idx against M0 and B4r (6b references, reproduced above) and against each other. Same 18 outer sets, 5 seeds for the MLP, 1 for the forest. **POST-HOC VARIANT, NOT A GATE RESULT.**

In [7]:
MODELS = ["M0", "M0+idx", "B4r", "B4r+idx"]; FAM = {"M0": "MLP", "M0+idx": "MLP", "B4r": "tree", "B4r+idx": "tree"}
print(f"*** {LABEL} ***  (M0+idx and B4r+idx add log cycle number; the frozen gate verdict W300 FAIL / W240 PASS is unchanged and is not recomputed)\n")
for w in WINS:
    print(f"=== Protocol D {w}: M0+idx (fixed config, 5 seeds, 18 outer sets) ==="); use_window(w)
    fr = run_deep(w, VAR_IDX, SETS, "pd"); PD_ARR[(w, "M0+idx")] = ens_array(fr, len(SEEDS), f"{w} M0+idx"); PD_SRC[(w, "M0+idx")] = "06c run"
    gc.collect(); torch.cuda.empty_cache()
need(all(PD_ARR[(w, m)].shape == (R, N) and not np.isnan(PD_ARR[(w, m)]).any() for w in WINS for m in MODELS), f"84 cells and {N:,} rows, 3 repeats, no NaN: all four models, both windows")

# ---- metrics
PDM, pd_rows, pdc_rows = {}, [], []
for w in WINS:
    for m in MODELS:
        P = PD_ARR[(w, m)]; vm = variant_metrics(P); cell = vm["cell"]; bias_c = seg_mean(P - Y).mean(0); PDM[(w, m)] = dict(vm, bias_cell=bias_c)
        pd_rows.append(dict(window=w, model=m, family=FAM[m], source=PD_SRC[(w, m)], macro_mae=vm["macro"], macro_excl_stitched=vm["macro_excl_stitched"], last20_macro_mae=vm["last20_macro"], pooled_bias=vm["bias"],
                            mae_life_short=float(cell[TERT == 0].mean()), mae_life_mid=float(cell[TERT == 1].mean()), mae_life_long=float(cell[TERT == 2].mean()),
                            gate_target=GATE_TARGET[w], macro_minus_target=vm["macro"] - GATE_TARGET[w], macro_over_target=vm["macro"] / GATE_TARGET[w], label=LABEL))
        for c in range(NC):
            pdc_rows.append(dict(window=w, model=m, cell_id=cells[c], batch=int(BATCH[c]), life=LIFE[c], stitched=bool(IS_ST[c]), life_tertile=TERT_LAB[TERT[c]], mae=cell[c], mean_signed_err=bias_c[c]))
PD_DF, PDC_DF = pd.DataFrame(pd_rows), pd.DataFrame(pdc_rows); save_csv(PD_DF, "protocol_d_models.csv"); save_csv(PDC_DF, "protocol_d_per_cell.csv")
print(f"Protocol D, cell-macro MAE (pp), mean of 3 repeats; {LABEL}. Gate-target columns are for information only (W300 {GATE_TARGET['W300']}, W240 {GATE_TARGET['W240']}); the gate verdict is frozen.")
print(PD_DF.drop(columns=["family", "label"]).to_string(index=False))

# ---- paired cluster-bootstrap contrasts (negative = first model has lower MAE)
PD_CONTRASTS = [("M0+idx", "M0", "MLP: +idx vs no-idx"), ("B4r+idx", "B4r", "tree: +idx vs no-idx"), ("M0+idx", "B4r+idx", "MLP+idx vs tree+idx"), ("M0", "B4r", "MLP vs tree, no idx (reference)")]
SUBS = {"all84": np.ones(NC, bool), "excl_stitched": ~IS_ST}; pc_rows = []
for w in WINS:
    for a, b, lab in PD_CONTRASTS:
        for sname, mask in SUBS.items():
            ca, cb = PDM[(w, a)]["cell"][mask], PDM[(w, b)]["cell"][mask]; d, lo, hi = paired(ca - cb, "3 Protocol D paired intervals")
            pc_rows.append(dict(window=w, contrast=lab, first=a, second=b, subset=sname, n_cells=int(mask.sum()), mae_first=float(ca.mean()), mae_second=float(cb.mean()), ratio=float(ca.mean() / cb.mean()),
                                diff=d, ci95_lo=lo, ci95_hi=hi, ci_excludes_zero=bool(lo > 0 or hi < 0), label=LABEL))
PDK_DF = pd.DataFrame(pc_rows); save_csv(PDK_DF, "protocol_d_contrasts.csv")
print(f"\nPaired cluster bootstrap over cells (5000 draws, seed 42), {INTERVALS['3 Protocol D paired intervals']} intervals; some will exclude zero by chance. {LABEL}.")
print(PDK_DF.drop(columns=["first", "second", "label"]).to_string(index=False))


*** POST-HOC VARIANT, NOT A GATE RESULT ***  (M0+idx and B4r+idx add log cycle number; the frozen gate verdict W300 FAIL / W240 PASS is unchanged and is not recomputed)

=== Protocol D W300: M0+idx (fixed config, 5 seeds, 18 outer sets) ===
[PASS] delta-Q check W300 row 36631
[PASS] delta-Q check W300 row 4994
[PASS] delta-Q check W300 row 43312
[PASS] sub_idx gives 32 grid points | 32
RAM available 2.7 GB (W300); top 8: python.exe 1.2GB, Code.exe 0.8GB, Code.exe 0.7GB, MemCompression 0.7GB, chrome.exe 0.6GB, Code.exe 0.5GB, chrome.exe 0.5GB, claude.exe 0.4GB
[PASS] count_params M0+idx | 3013 vs 3013
[PASS] no NaN/inf predictions: W300/idx
  trained W300/idx: 90 members in 76s
=== Protocol D W240: M0+idx (fixed config, 5 seeds, 18 outer sets) ===
[PASS] delta-Q check W240 row 36631
[PASS] delta-Q check W240 row 4994
[PASS] delta-Q check W240 row 43312
[PASS] sub_idx gives 32 grid points | 32
RAM available 2.4 GB (W240); top 8: python.exe 1.9GB, Code.exe 0.7GB, MemCompression 0.7GB, chr

## 4. Slow-ager shift tests
(a) extreme-life holdout exactly as in 6b; (b) harder shift: train on life <= 710, test on life > 710. Policy twins of test cells are removed from training. Frozen config, 5 seeds for the MLP.

In [8]:
# ---- 4a: extreme-life holdout exactly as in 6b (separate short / long groups; policy twins of held-out cells removed from training)
K_EXT = int(round(0.10 * NC)); _ls = np.sort(LIFE); short_cut, long_cut = _ls[K_EXT - 1], _ls[-K_EXT]
SHORT = [c for c in cells if LIFE[CIDX[c]] <= short_cut]; LONG = [c for c in cells if LIFE[CIDX[c]] >= long_cut]
print(f"held-out groups: shortest {len(SHORT)} cells (life <= {short_cut:.0f}), longest {len(LONG)} cells (life >= {long_cut:.0f})")
need(8 <= len(SHORT) <= 9 and 8 <= len(LONG) <= 9, "8 or 9 cells in each extreme group", (len(SHORT), len(LONG)))
def make_set(name, gi, held, train_pool):
    hp_ = {POLICY[c] for c in held}; twins = sorted(c for c in train_pool if c not in held and POLICY[c] in hp_); train = [c for c in train_pool if c not in held and c not in twins]
    need(not set(train) & set(held) and not set(train) & set(twins), f"{name}: train and test cells disjoint, no twin in training")
    need(not any(POLICY[c] in hp_ for c in train), f"{name}: no training cell shares a policy_norm with a test cell")
    need(not (set(train) | set(held)) & SENS_CELLS and not any(c.startswith("b3") for c in train + held), f"{name}: no batch-3 / sensitivity cells")
    return dict(repeat=0, outer_fold=gi, train=train, test=sorted(held), name=name), twins, hp_
EXT_SETS, SHIFT_INFO, TWINS = [], [], {}
for gi, (gname, held) in enumerate((("short", SHORT), ("long", LONG))):
    s, twins, hp_ = make_set(f"extreme_{gname}", gi, held, list(cells)); EXT_SETS.append(s); TWINS[s["name"]] = twins
# ---- 4b: harder shift: train on life <= 710 only (lower two tertiles), test on life > 710 (longest tertile)
HARD_TEST = [c for c in cells if LIFE[CIDX[c]] > TERT_CUTS[1]]; HARD_POOL = [c for c in cells if LIFE[CIDX[c]] <= TERT_CUTS[1]]
need(set(HARD_TEST) == {cells[i] for i in np.where(TERT == 2)[0]}, "life > 710 cells equal the longest life tertile of 6b (cuts 493 / 710)", level="WARN")
need(len(HARD_TEST) == 28, "28 cells in the longest tertile", len(HARD_TEST), level="WARN")
s, twins, hp_ = make_set("longest_tertile", 0, HARD_TEST, HARD_POOL); HARD_SETS = [s]; TWINS[s["name"]] = twins
need(not any(LIFE[CIDX[c]] > TERT_CUTS[1] for c in s["train"]), "longest_tertile: every training cell has life <= 710")
ALL_SETS = EXT_SETS + HARD_SETS
for s in ALL_SETS:
    tl = LIFE[[CIDX[c] for c in s["train"]]]; nst = int(sum(c in STITCHED for c in s["test"]))
    SHIFT_INFO.append(dict(test=s["name"], n_train=len(s["train"]), n_test=len(s["test"]), n_test_stitched=nst, n_test_excl_stitched=len(s["test"]) - nst, n_policies_test=len({POLICY[c] for c in s["test"]}),
                           n_twins_removed=len(TWINS[s["name"]]), train_life_min=float(tl.min()), train_life_max=float(tl.max()), test_life_min=float(LIFE[[CIDX[c] for c in s["test"]]].min()),
                           test_life_max=float(LIFE[[CIDX[c] for c in s["test"]]].max()), twins_removed=" ".join(TWINS[s["name"]])))
    print(f"  {s['name']}: test {len(s['test'])} cells ({nst} stitched), removed {len(TWINS[s['name']])} policy twins, trained on {len(s['train'])} cells (life {tl.min():.0f}-{tl.max():.0f})")
SHIFT_INFO_DF = pd.DataFrame(SHIFT_INFO); save_csv(SHIFT_INFO_DF, "shift_design.csv")
# same held-out cells and twins as 6b
if P_EXTCELLS.exists() and P_EXTCOUNTS.exists():
    _ec, _cn = pd.read_csv(P_EXTCELLS), pd.read_csv(P_EXTCOUNTS).fillna("")
    for s in EXT_SETS:
        g = s["name"].replace("extreme_", "")
        for w in WINS: need(set(_ec[(_ec.window == w) & (_ec.group == g)].cell_id) == set(s["test"]), f"6b extreme holdout: same held-out cells ({g}, {w})")
        r6 = _cn[_cn.group == g].iloc[0]; need(int(r6.n_twins_removed) == len(TWINS[s["name"]]) and int(r6.n_train) == len(s["train"]) and sorted(str(r6.twins_removed).split()) == TWINS[s["name"]], f"6b extreme holdout: same policy twins removed and training size ({g})", f"twins {TWINS[s['name']]}, n_train {len(s['train'])}")
else:
    _ec = None; chk("WARN", "6b extreme_holdout_cells.csv / counts not found; held-out sets not cross-checked")

# ---- model runs (6b cache reused for the no-index models of the extreme holdout when valid; everything else is trained here)
SHIFT_PRED, SHIFT_SRC = {}, {}
def put(w, s, model, fr, ns, src):
    v = ens_vec(fr[(fr["repeat"] == s["repeat"]) & (fr["outer_fold"] == s["outer_fold"])], ns, f"{w} {s['name']} {model}")
    need(set(CELLID[~np.isnan(v)]) == set(s["test"]), f"{w} {s['name']} {model}: predictions cover exactly the test cells"); SHIFT_PRED[(w, s["name"], model)] = v; SHIFT_SRC[(w, s["name"], model)] = src
for w in WINS:
    print(f"\n=== shift tests {w} ==="); use_window(w); tseed = [COMMON["tree_seed"]]
    fm, pm = load_6b(f"ext_deep_{w}_ref_*.parquet", EXT_SETS, SEEDS); ft, pt = load_6b(f"ext_tree_{w}_ext_B4r_*.parquet", EXT_SETS, tseed)
    srcm, srct = (f"6b cache: {pm.name}" if fm is not None else "re-run"), (f"6b cache: {pt.name}" if ft is not None else "re-run")
    if fm is None: fm = run_deep(w, VAR_REF, EXT_SETS, "ext")
    if ft is None: ft = run_tree(w, "ext_B4r", False, EXT_SETS, "ext")
    fmi = run_deep(w, VAR_IDX, EXT_SETS, "ext"); fti = run_tree(w, "ext_idx", True, EXT_SETS, "ext")
    for s in EXT_SETS:
        put(w, s, "M0", fm, len(SEEDS), srcm); put(w, s, "B4r", ft, 1, srct); put(w, s, "M0+idx", fmi, len(SEEDS), "06c run"); put(w, s, "B4r+idx", fti, 1, "06c run")
    hs = HARD_SETS[0]
    for model, fr_, ns in (("M0", run_deep(w, VAR_REF, HARD_SETS, "hard"), len(SEEDS)), ("M0+idx", run_deep(w, VAR_IDX, HARD_SETS, "hard"), len(SEEDS)),
                           ("B4r", run_tree(w, "hard_B4r", False, HARD_SETS, "hard"), 1), ("B4r+idx", run_tree(w, "hard_idx", True, HARD_SETS, "hard"), 1)):
        put(w, hs, model, fr_, ns, "06c run")
    gc.collect(); torch.cuda.empty_cache()
# reproduce the 6b extreme-holdout per-cell MAEs of M0 and B4r
if _ec is not None:
    for w in WINS:
        for s in EXT_SETS:
            for model, col in (("M0", "M0_extreme_mae"), ("B4r", "B4r_extreme_mae")):
                ref = _ec[(_ec.window == w) & (_ec.group == s["name"].replace("extreme_", ""))].set_index("cell_id")[col]; P = SHIFT_PRED[(w, s["name"], model)]
                dmax = max(abs(float(np.abs(P[CELL_POS[c]] - Y[CELL_POS[c]]).mean()) - float(ref[c])) for c in s["test"])
                repro_gate(dmax, SHIFT_SRC[(w, s["name"], model)].startswith("6b"), "tree" if model == "B4r" else "deep", f"reproduces 6b extreme-holdout per-cell MAE {w} {s['name']} {model}", f"max |diff| {dmax:.1e}")

# ---- per-cell and summary tables
D_CELL = {(w, m): dict(mae=PDM[(w, m)]["cell"], bias=PDM[(w, m)]["bias_cell"]) for w in WINS for m in MODELS}
sc_rows = []
for w in WINS:
    for s in ALL_SETS:
        tl = LIFE[[CIDX[c] for c in s["train"]]]
        for m in MODELS:
            P = SHIFT_PRED[(w, s["name"], m)]
            for c in s["test"]:
                ci_ = CIDX[c]; sl = CELL_POS[c]; e = P[sl] - Y[sl]
                sc_rows.append(dict(window=w, test=s["name"], model=m, cell_id=c, batch=int(BATCH[ci_]), stitched=bool(IS_ST[ci_]), life=LIFE[ci_], life_over_train_max=max(0.0, LIFE[ci_] - tl.max()),
                                    mae=float(np.abs(e).mean()), mean_signed_err=float(e.mean()), protocol_d_mae=float(D_CELL[(w, m)]["mae"][ci_]), protocol_d_bias=float(D_CELL[(w, m)]["bias"][ci_]), source=SHIFT_SRC[(w, s["name"], m)]))
SHIFT_CELL = pd.DataFrame(sc_rows); save_csv(SHIFT_CELL, "shift_per_cell.csv")
sm_rows = []
for (w, t, m), g in SHIFT_CELL.groupby(["window", "test", "model"], sort=False):
    info = SHIFT_INFO_DF[SHIFT_INFO_DF.test == t].iloc[0]
    for sname, gg in (("all", g), ("excl_stitched", g[~g.stitched])):
        sm_rows.append(dict(window=w, test=t, subset=sname, model=m, n_train=int(info.n_train), n_test=len(gg), n_twins_removed=int(info.n_twins_removed), mae=gg.mae.mean(), mean_signed_err=gg.mean_signed_err.mean(),
                            protocol_d_mae=gg.protocol_d_mae.mean(), ratio_to_protocol_d=gg.mae.mean() / gg.protocol_d_mae.mean(), protocol_d_bias=gg.protocol_d_bias.mean(), label=LABEL))
SHIFT_SUM = pd.DataFrame(sm_rows); save_csv(SHIFT_SUM, "shift_results.csv")
print(f"\nShift tests; MAE = cell-macro MAE (pp) on the test cells; signed error + = SOH over-estimated; ratio = MAE / the same model's own Protocol D out-of-fold MAE on the same cells. {LABEL}.")
print(SHIFT_SUM.drop(columns="label").to_string(index=False))
print("\nPer cell (first rows):"); print(SHIFT_CELL.drop(columns="source").head(12).to_string(index=False))


held-out groups: shortest 8 cells (life <= 457), longest 8 cells (life >= 1051)
[PASS] 8 or 9 cells in each extreme group | (8, 8)
[PASS] extreme_short: train and test cells disjoint, no twin in training
[PASS] extreme_short: no training cell shares a policy_norm with a test cell
[PASS] extreme_short: no batch-3 / sensitivity cells
[PASS] extreme_long: train and test cells disjoint, no twin in training
[PASS] extreme_long: no training cell shares a policy_norm with a test cell
[PASS] extreme_long: no batch-3 / sensitivity cells
[PASS] life > 710 cells equal the longest life tertile of 6b (cuts 493 / 710)
[PASS] 28 cells in the longest tertile | 28
[PASS] longest_tertile: train and test cells disjoint, no twin in training
[PASS] longest_tertile: no training cell shares a policy_norm with a test cell
[PASS] longest_tertile: no batch-3 / sensitivity cells
[PASS] longest_tertile: every training cell has life <= 710
  extreme_short: test 8 cells (0 stitched), removed 0 policy twins, trained

## 5. Summary
+idx against no-idx for each shift test, window and model family. The one-line findings are generated from the numbers.

In [9]:
# ---- paired contrasts: +idx vs no-idx per shift test, window, subset and model family (cells shared by construction)
SCI = SHIFT_CELL.set_index(["window", "test", "model", "cell_id"])
FAMS = {"MLP": ("M0+idx", "M0"), "tree": ("B4r+idx", "B4r")}
rows = []
for w in WINS:
    for s in ALL_SETS:
        for sname in ("all", "excl_stitched"):
            cs_ = [c for c in s["test"] if sname == "all" or c not in STITCHED]
            if not cs_ or (sname == "excl_stitched" and len(cs_) == len(s["test"])): continue
            ix = [CIDX[c] for c in cs_]
            for fam, (a, b) in FAMS.items():
                sa = np.array([SCI.loc[(w, s["name"], a, c), "mae"] for c in cs_]); sb = np.array([SCI.loc[(w, s["name"], b, c), "mae"] for c in cs_])
                da, db = D_CELL[(w, a)]["mae"][ix], D_CELL[(w, b)]["mae"][ix]
                d, lo, hi = paired(sa - sb, "4 shift paired intervals"); dD, loD, hiD = paired(da - db, "4 shift paired intervals (same cells, Protocol D)")
                ba = np.array([SCI.loc[(w, s["name"], a, c), "mean_signed_err"] for c in cs_]); bb = np.array([SCI.loc[(w, s["name"], b, c), "mean_signed_err"] for c in cs_])
                rows.append(dict(window=w, test=s["name"], subset=sname, family=fam, n_cells=len(cs_), mae_noidx=sb.mean(), mae_idx=sa.mean(), ratio_idx_over_noidx=sa.mean() / sb.mean(), diff=d, ci95_lo=lo, ci95_hi=hi,
                                 protocol_d_mae_noidx=db.mean(), protocol_d_mae_idx=da.mean(), protocol_d_ratio=da.mean() / db.mean(), protocol_d_diff=dD, protocol_d_ci95_lo=loD, protocol_d_ci95_hi=hiD,
                                 growth_noidx=sb.mean() / db.mean(), growth_idx=sa.mean() / da.mean(), bias_noidx=bb.mean(), bias_idx=ba.mean()))
for _, r in PDK_DF[PDK_DF["first"].isin(["M0+idx", "B4r+idx"]) & PDK_DF["second"].isin(["M0", "B4r"]) & (PDK_DF["first"].str.replace("+idx", "", regex=False) == PDK_DF["second"])].iterrows():
    fam = "MLP" if r["first"] == "M0+idx" else "tree"; vm_a, vm_b = PDM[(r.window, r["first"])], PDM[(r.window, r["second"])]; mask = SUBS[r.subset]
    rows.append(dict(window=r.window, test="protocol_D", subset="all" if r.subset == "all84" else "excl_stitched", family=fam, n_cells=int(r.n_cells), mae_noidx=r.mae_second, mae_idx=r.mae_first, ratio_idx_over_noidx=r.ratio,
                     diff=r["diff"], ci95_lo=r.ci95_lo, ci95_hi=r.ci95_hi, bias_noidx=float(vm_b["bias_cell"][mask].mean()), bias_idx=float(vm_a["bias_cell"][mask].mean())))
TEST_ORDER = {"protocol_D": 0, "extreme_short": 1, "extreme_long": 2, "longest_tertile": 3}
SUMMARY = pd.DataFrame(rows); SUMMARY["_o"] = SUMMARY.test.map(TEST_ORDER); SUMMARY["_s"] = (SUMMARY.subset != "all").astype(int)
SUMMARY = SUMMARY.sort_values(["window", "_o", "_s", "family"], ascending=True).drop(columns=["_o", "_s"]).reset_index(drop=True); SUMMARY["label"] = LABEL
save_csv(SUMMARY, "summary_idx_vs_noidx.csv")

def word(lo, hi): return "lower" if hi < 0 else "higher" if lo > 0 else "no clear difference"
FIND = []
for (w, t, sn), g in SUMMARY.groupby(["window", "test", "subset"], sort=False):
    g = g.set_index("family"); n = int(g.n_cells.iloc[0]); parts = []
    for fam in ("MLP", "tree"):
        r = g.loc[fam]; s_ = f"{fam}: +idx/no-idx MAE ratio {r.ratio_idx_over_noidx:.2f} ({word(r.ci95_lo, r.ci95_hi)}; diff {r['diff']:+.3f} pp [{r.ci95_lo:+.3f}, {r.ci95_hi:+.3f}])"
        if t != "protocol_D":
            s_ += f"; same cells in Protocol D ratio {r.protocol_d_ratio:.2f} (diff {r.protocol_d_diff:+.3f} [{r.protocol_d_ci95_lo:+.3f}, {r.protocol_d_ci95_hi:+.3f}]); error vs own Protocol D x{r.growth_idx:.2f} with idx, x{r.growth_noidx:.2f} without"
        parts.append(s_)
    FIND.append(dict(window=w, test=t, subset=sn, n_cells=n, finding=f"{w} | {t} | {sn} | {n} cells | " + " | ".join(parts)))
FIND_DF = pd.DataFrame(FIND); save_csv(FIND_DF, "summary_findings.csv")
print(f"{LABEL}. Ratio < 1 = +idx has lower MAE. 'growth' = shift MAE divided by the same model's Protocol D out-of-fold MAE on the same cells. Intervals: paired cluster bootstrap over the listed cells (small n for several rows).\n")
print(SUMMARY.drop(columns="label").to_string(index=False)); print(); [print(f) for f in FIND_DF.finding]
print("\nintervals so far:", INTERVALS)


POST-HOC VARIANT, NOT A GATE RESULT. Ratio < 1 = +idx has lower MAE. 'growth' = shift MAE divided by the same model's Protocol D out-of-fold MAE on the same cells. Intervals: paired cluster bootstrap over the listed cells (small n for several rows).

window            test        subset family  n_cells  mae_noidx  mae_idx  ratio_idx_over_noidx    diff  ci95_lo  ci95_hi  protocol_d_mae_noidx  protocol_d_mae_idx  protocol_d_ratio  protocol_d_diff  protocol_d_ci95_lo  protocol_d_ci95_hi  growth_noidx  growth_idx  bias_noidx  bias_idx
  W240      protocol_D           all    MLP       84     0.5224   0.4436                0.8491 -0.0788  -0.1324  -0.0232                   NaN                 NaN               NaN              NaN                 NaN                 NaN           NaN         NaN      0.0838    0.0893
  W240      protocol_D           all   tree       84     0.5212   0.4425                0.8489 -0.0787  -0.1090  -0.0478                   NaN                 NaN               

## 6. Figures, report, final checks

In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 10})
COL = {"M0": "#0072B2", "M0+idx": "#D55E00", "B4r": "#009E73", "B4r+idx": "#CC79A7"}
def save_fig(fig, name):
    p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p

# (a) paired forest plot: +idx minus no-idx in cell-macro MAE, per test / window / subset / family; open diamonds = the same cells in Protocol D
S = SUMMARY.reset_index(drop=True); yy = np.arange(len(S))[::-1]
fig, ax = plt.subplots(figsize=(9, 0.26 * len(S) + 1.8))
for y_, r in zip(yy, S.itertuples()):
    c = COL["M0+idx"] if r.family == "MLP" else COL["B4r+idx"]; ax.errorbar(r.diff, y_, xerr=[[r.diff - r.ci95_lo], [r.ci95_hi - r.diff]], fmt="o" if r.subset == "all" else "s", mfc=c if r.subset == "all" else "white", color=c, capsize=2, ms=4.5)
    if r.test != "protocol_D": ax.plot(r.protocol_d_diff, y_ - 0.28, "d", mfc="white", mec="grey", ms=4)
ax.axvline(0, color="k", lw=0.8); ax.set_yticks(yy); ax.set_yticklabels([f"{r.window} {r.test} {r.subset} {r.family} (n={r.n_cells})" for r in S.itertuples()], fontsize=6.5)
ax.set_xlabel("MAE(+idx) - MAE(no idx), pp; negative = +idx lower; 95% paired cluster bootstrap. open diamond = same cells in Protocol D")
ax.set_title(f"+idx vs no-idx per test ({len(S)} contrasts; {LABEL})", fontsize=9)
save_fig(fig, "fig_a_forest_idx_vs_noidx.png")

# (b) per-cell signed error vs life, harder shift, M0 and M0+idx
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=False)
for ax, w in zip(axs, WINS):
    g = SHIFT_CELL[(SHIFT_CELL.window == w) & (SHIFT_CELL.test == "longest_tertile")]; a, b = g[g.model == "M0"].set_index("cell_id"), g[g.model == "M0+idx"].set_index("cell_id").loc[g[g.model == "M0"].cell_id]
    a = a.loc[b.index]
    for c in b.index: ax.plot([a.loc[c, "life"]] * 2, [a.loc[c, "mean_signed_err"], b.loc[c, "mean_signed_err"]], color="grey", lw=0.6, alpha=0.6)
    ax.scatter(a.life, a.mean_signed_err, s=24, color=COL["M0"], label="M0 (no idx)"); ax.scatter(b.life, b.mean_signed_err, s=24, marker="s", color=COL["M0+idx"], label="M0+idx")
    st = a.stitched.values; ax.scatter(a.life[st], a.mean_signed_err[st], s=70, facecolor="none", edgecolor="k", lw=1.0, label="stitched (b1c0-4)")
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("cell life (cycles)"); ax.set_title(f"{w}: longest tertile (train life <= 710), MAE {a.mae.mean():.2f} vs {b.mae.mean():.2f}", fontsize=8)
axs[0].set_ylabel("mean signed error (pp)  [+ = over-estimated]"); axs[0].legend(frameon=False, fontsize=7)
save_fig(fig, "fig_b_signed_error_vs_life_longest_tertile.png")

# (c) MAE by test and model, markers = each model's own Protocol D MAE on the same cells
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True); tests = [s["name"] for s in ALL_SETS]
for ax, w in zip(axs, WINS):
    t = SHIFT_SUM[(SHIFT_SUM.window == w) & (SHIFT_SUM.subset == "all")]; x = np.arange(len(tests)); wd = 0.2
    for k, m in enumerate(MODELS):
        tm = t[t.model == m].set_index("test").loc[tests]; xs = x + (k - 1.5) * wd
        ax.bar(xs, tm.mae, wd, color=COL[m], label=m); ax.plot(xs, tm.protocol_d_mae, "_", color="k", ms=11, mew=1.6, label="own Protocol D MAE, same cells" if k == 0 else None)
    ax.set_xticks(x); ax.set_xticklabels([f"{s['name'].replace('_', ' ')}\n(n={len(s['test'])})" for s in ALL_SETS], fontsize=8); ax.set_title(f"{w}: cell-macro MAE on the test cells")
axs[0].set_ylabel("MAE (pp)"); _h, _l = axs[0].get_legend_handles_labels(); fig.legend(_h, _l, loc="lower center", ncol=5, frameon=False, fontsize=7, bbox_to_anchor=(0.5, -0.07))
save_fig(fig, "fig_c_mae_by_test_and_model.png")
for p in sorted(FIG.glob("*.png")): need(p.stat().st_size > 0, f"figure written: {p.name}", f"{p.stat().st_size / 1024:.0f} KB")


C:\Users\kulje\AppData\Local\Temp\ipykernel_12780\927738483.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p
C:\Users\kulje\AppData\Local\Temp\ipykernel_12780\927738483.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p


[PASS] figure written: fig_a_forest_idx_vs_noidx.png | 144 KB
[PASS] figure written: fig_b_signed_error_vs_life_longest_tertile.png | 71 KB
[PASS] figure written: fig_c_mae_by_test_and_model.png | 39 KB


C:\Users\kulje\AppData\Local\Temp\ipykernel_12780\927738483.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  p = _guard(FIG / name); fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); WRITTEN.append(p); plt.show(); plt.close(fig); return p


In [11]:
# ---- final checks
def _no_nan(df, cols, label):
    bad = int(df[cols].isna().sum().sum()); need(bad == 0, f"no NaN in table: {label}", f"{bad} NaN cells")
_no_nan(REPRO, ["recomputed", "ablation_results", "abs_diff"], "reproduction_check"); _no_nan(PD_DF, ["macro_mae", "macro_excl_stitched", "last20_macro_mae", "pooled_bias", "mae_life_short", "mae_life_mid", "mae_life_long"], "protocol_d_models")
_no_nan(PDC_DF, ["mae", "mean_signed_err"], "protocol_d_per_cell"); _no_nan(PDK_DF, ["mae_first", "mae_second", "ratio", "diff", "ci95_lo", "ci95_hi"], "protocol_d_contrasts")
_no_nan(SHIFT_CELL, ["mae", "mean_signed_err", "protocol_d_mae", "protocol_d_bias"], "shift_per_cell"); _no_nan(SHIFT_SUM, ["mae", "mean_signed_err", "protocol_d_mae", "ratio_to_protocol_d"], "shift_results")
_no_nan(SUMMARY, ["mae_noidx", "mae_idx", "ratio_idx_over_noidx", "diff", "ci95_lo", "ci95_hi"], "summary_idx_vs_noidx"); _no_nan(SUMMARY[SUMMARY.test != "protocol_D"], ["protocol_d_ratio", "protocol_d_diff", "growth_idx", "growth_noidx"], "summary_idx_vs_noidx (shift rows)")
need(all(PD_ARR[(w, m)].shape == (R, N) for w in WINS for m in MODELS) and N == EXPECTED_ROWS and NC == EXPECTED_CELLS and len(cells) == 84, f"84 cells and {EXPECTED_ROWS:,} rows in every full-protocol result", f"{len(PD_ARR)} arrays")
_all_cells = set(cells) | set(PDC_DF.cell_id) | set(SHIFT_CELL.cell_id) | {c for s in ALL_SETS for c in s["train"] + s["test"]}
need(not (_all_cells & SENS_CELLS) and not any(str(c).startswith("b3") for c in _all_cells), "no batch-3 / sensitivity cells in any array, set or table")
need(all(not set(s["train"]) & set(s["test"]) for s in SETS + ALL_SETS), "train and test cells disjoint in every set (18 Protocol D outer sets and the 3 shift tests)",
     f"also asserted per member at training time ({MEMBER_AUDIT['members']} deep members, {MEMBER_AUDIT['trees']} forests this run)")
need(all(not any(POLICY[c] in {POLICY[t] for t in s["test"]} for c in s["train"]) for s in ALL_SETS), "no policy twin of a test cell in any shift-test training set")
need(all(max(LIFE[CIDX[c]] for c in s["train"]) <= TERT_CUTS[1] for s in HARD_SETS), "longest-tertile training cells all have life <= 710")
need(len(SHIFT_CELL) == len(WINS) * len(MODELS) * sum(len(s["test"]) for s in ALL_SETS), "shift_per_cell has one row per window x model x test cell")
need(all(OUT.resolve() in Path(p).resolve().parents for p in WRITTEN), "all outputs written under outputs/phase6c only (plus src/soh_deep_v3.py, written in section 2)", f"{len(WRITTEN)} files")
HASH1 = hash_state()
for _k in ("gate", "module", "verdict", "frozen_config"):
    need(HASH1[_k]["sha"] == HASH0[_k]["sha"] == HASH1[_k]["exp"], f"sha256 unchanged at end: {_k}", HASH1[_k]["sha"][:16] + "...")
    need(HASH1[_k]["mtime"] == HASH0[_k]["mtime"], f"mtime unchanged at end: {_k}")
need(sha_raw(P_VERDICT) == VERDICT_SHA, "verdict file untouched (W300 FAIL / W240 PASS, never recomputed)")
need(sha_lf(P_MODULE3) == V3_SHA, "soh_deep_v3.py unchanged at end", V3_SHA[:16] + "...")
TOTAL_INTERVALS = int(sum(INTERVALS.values()))
print("\nbootstrap intervals computed in this phase:", INTERVALS, "| total", TOTAL_INTERVALS)
chk("PASS", "total interval count printed", TOTAL_INTERVALS)

# ---- fixed-width report (style of phase6b_report.txt)
def tbl(df, cols, fm=None, hdr=None):
    fm = fm or {}; hdr = hdr or {}
    cell = lambda c, v: "-" if (isinstance(v, float) and np.isnan(v)) else (fm[c](v) if c in fm else (f"{v:.3f}" if isinstance(v, (float, np.floating)) else str(v)))
    rows = [[cell(c, v) for c, v in zip(cols, r)] for r in df[cols].itertuples(index=False, name=None)]; heads = [hdr.get(c, c) for c in cols]
    wd = [max(len(h), *(len(r[i]) for r in rows)) if rows else len(h) for i, h in enumerate(heads)]
    num = [pd.api.types.is_numeric_dtype(df[c]) and df[c].dtype != bool for c in cols]; fr_ = lambda r: "  ".join(x.rjust(wd[i]) if num[i] else x.ljust(wd[i]) for i, x in enumerate(r))
    return "\n".join([fr_(heads), "  ".join("-" * x for x in wd)] + [fr_(r) for r in rows])
sg = lambda v: f"{v:+.3f}"; f4 = lambda v: f"{v:.4f}"
bar = "=" * 110; L = []; sec = lambda t: L.extend(["", bar, t, bar])
L += [bar, f"PHASE 6c  CYCLE-INDEX SHIFT TEST  ({LABEL}; nothing here is a gate result)", bar,
      f"generated {dt.datetime.now():%Y-%m-%d %H:%M}   root {ROOT}   runtime {(time.time() - T_START) / 60:.1f} min   GPU {GPU_INFO}",
      f"gate sha256 {HASH1['gate']['sha']}", f"soh_deep_v2 sha256 (CRLF->LF) {HASH1['module']['sha']}  (unchanged)", f"soh_deep_v3 sha256 (LF) {V3_SHA}",
      f"gate_verdict sha256 {HASH1['verdict']['sha']}  (frozen: W300 FAIL, W240 PASS; never recomputed)", f"outputs/phase6b/frozen_config.json sha256 {FZ_SHA}  (used as is; nothing tuned)",
      f"development rows {N:,} | cells {NC} | repeats {R} | err = pred - y (+ = over-estimated) | bootstrap: cells, {B_BOOT} draws, seed {SEED_BOOT}",
      f"total bootstrap intervals computed in this phase: {TOTAL_INTERVALS}  {INTERVALS}",
      "+idx = the model also sees the row's own log cycle number (MLP: standardised log1p(cycle) from the training rows, 34 inputs, 3,013 parameters; tree: log(cycle) as in 6b B4r:+log_cycle).",
      "no-idx numbers: M0 / B4r = 6b fixed-config references (reproduced below). Batch-3 and the 5 sensitivity cells never appear."]
sec("1. REPRODUCTION OF THE 6b REFERENCE ROWS (must match ablation_results.csv within 1e-6)")
L.append(tbl(REPRO, ["window", "model", "variant_6b", "metric", "recomputed", "ablation_results", "abs_diff", "source"], {"recomputed": lambda v: f"{v:.6f}", "ablation_results": lambda v: f"{v:.6f}", "abs_diff": lambda v: f"{v:.1e}"}))
sec("2. soh_deep_v3.py (v2 + one optional extra scalar input; v2 untouched)")
L += [f"sha256 {V3_SHA}", f"{len(PATCH_LOG)} exact-match replacements applied to v2; diff lines: +{sum(l.startswith('+') and not l.startswith('+++') for l in _d)} / -{sum(l.startswith('-') and not l.startswith('---') for l in _d)}", "", tbl(V3_CHECKS, ["level", "name", "detail"])]
sec(f"3. PROTOCOL D  ({LABEL}); gate-target columns are for information only")
L.append(tbl(PD_DF, ["window", "model", "macro_mae", "macro_excl_stitched", "last20_macro_mae", "pooled_bias", "mae_life_short", "mae_life_mid", "mae_life_long", "gate_target", "macro_minus_target", "macro_over_target"],
             {"pooled_bias": sg, "macro_minus_target": sg, "gate_target": f4}, {"mae_life_short": "mae_short", "mae_life_mid": "mae_mid", "mae_life_long": "mae_long", "macro_minus_target": "minus_target", "macro_over_target": "over_target"}))
L += ["", f"life tertiles: cuts 493 / 710 ({int((TERT == 0).sum())} / {int((TERT == 1).sum())} / {int((TERT == 2).sum())} cells). Paired contrasts (negative diff = first model lower MAE), cluster bootstrap over cells:",
      tbl(PDK_DF.assign(ci95=[f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(PDK_DF.ci95_lo, PDK_DF.ci95_hi)]), ["window", "contrast", "subset", "n_cells", "mae_first", "mae_second", "ratio", "diff", "ci95"], {"diff": sg})]
sec("4. SLOW-AGER SHIFT TESTS (frozen config, 5 MLP seeds, policy twins of test cells removed from training)")
L.append(tbl(SHIFT_INFO_DF, ["test", "n_train", "n_test", "n_test_stitched", "n_test_excl_stitched", "n_twins_removed", "train_life_min", "train_life_max", "test_life_min", "test_life_max", "twins_removed"],
             {c: (lambda v: f"{v:.0f}") for c in ("train_life_min", "train_life_max", "test_life_min", "test_life_max")}))
L += ["", "MAE = cell-macro MAE on the test cells (pp); signed error + = over-estimated; ratio = MAE / the model's own Protocol D out-of-fold MAE on the same cells.",
      tbl(SHIFT_SUM, ["window", "test", "subset", "model", "n_train", "n_test", "n_twins_removed", "mae", "mean_signed_err", "protocol_d_mae", "ratio_to_protocol_d"], {"mean_signed_err": sg}, {"protocol_d_mae": "pD_mae", "ratio_to_protocol_d": "ratio_to_pD"})]
sec(f"5. SUMMARY: +idx vs no-idx  ({LABEL})")
L.append(tbl(SUMMARY.assign(ci95=[f"[{a:+.3f}, {b:+.3f}]" for a, b in zip(SUMMARY.ci95_lo, SUMMARY.ci95_hi)]), ["window", "test", "subset", "family", "n_cells", "mae_noidx", "mae_idx", "ratio_idx_over_noidx", "diff", "ci95", "protocol_d_ratio", "growth_noidx", "growth_idx"],
             {"diff": sg}, {"ratio_idx_over_noidx": "ratio", "protocol_d_ratio": "pD_ratio", "growth_noidx": "growth_no", "growth_idx": "growth_idx"}))
L += ["", "ratio < 1: +idx has lower MAE. pD_ratio = the same ratio on the same cells in Protocol D. growth = shift MAE / own Protocol D MAE on the same cells. Several rows rest on 3-8 cells; read the intervals.", ""] + list(FIND_DF.finding)
sec("6. CHECKS")
_cnt = {lv: sum(c["level"] == lv for c in CHECKS) for lv in ("PASS", "WARN", "FAIL")}; L.append(f"so far: {_cnt}")
L += [f"  [{c['level']}] {c['name']} | {c['detail']}" for c in CHECKS if c["level"] in ("WARN", "FAIL")]
_rp = save_text("\n".join(L) + "\n", "phase6c_report.txt")
def artifact_list():
    return [dict(file=str(p.relative_to(OUT)).replace("\\", "/"), bytes=p.stat().st_size) for p in sorted(OUT.rglob("*")) if p.is_file() and "cache" not in p.relative_to(OUT).parts]
dump_checks(partial=False, extra=dict(artifacts=artifact_list(), elapsed_seconds=round(time.time() - T_START, 1), hashes=dict(start=HASH0, end=HASH1), soh_deep_v3_sha256=V3_SHA, frozen_config_sha256=FZ_SHA, gpu=GPU_INFO))
keep_awake(False)
print(_rp.read_text(encoding="utf-8"))
print(f"\nFINAL CHECKS: PASS {sum(c['level'] == 'PASS' for c in CHECKS)} | WARN {sum(c['level'] == 'WARN' for c in CHECKS)} | FAIL {sum(c['level'] == 'FAIL' for c in CHECKS)} | intervals {TOTAL_INTERVALS}")
print("\nARTIFACTS (outputs/phase6c, cache excluded)")
print(pd.DataFrame(artifact_list()).assign(KB=lambda d: (d.bytes / 1024).round(1)).drop(columns="bytes").to_string(index=False))


[PASS] no NaN in table: reproduction_check | 0 NaN cells
[PASS] no NaN in table: protocol_d_models | 0 NaN cells
[PASS] no NaN in table: protocol_d_per_cell | 0 NaN cells
[PASS] no NaN in table: protocol_d_contrasts | 0 NaN cells
[PASS] no NaN in table: shift_per_cell | 0 NaN cells
[PASS] no NaN in table: shift_results | 0 NaN cells
[PASS] no NaN in table: summary_idx_vs_noidx | 0 NaN cells
[PASS] no NaN in table: summary_idx_vs_noidx (shift rows) | 0 NaN cells
[PASS] 84 cells and 55,963 rows in every full-protocol result | 8 arrays
[PASS] no batch-3 / sensitivity cells in any array, set or table
[PASS] train and test cells disjoint in every set (18 Protocol D outer sets and the 3 shift tests) | also asserted per member at training time (220 deep members, 8 forests this run)
[PASS] no policy twin of a test cell in any shift-test training set
[PASS] longest-tertile training cells all have life <= 710
[PASS] shift_per_cell has one row per window x model x test cell
[PASS] all outputs wri